# Notebook 03 — ASR Baseline Training (Cascaded RQ1)

**Scope:** Build and *pilot-validate* the **ASR baseline** of the Cascaded system:
`Bahnar audio → Bahnar transcript`. This notebook does **not** translate to
Vietnamese and does **not** touch the frozen RQ1 test set.

- Data prepared by Notebook 01–02 (clean train = 102,507; clean validation = 11,113).
- Frozen test (215 records) is **never** read, trained, evaluated, or tuned on here.
- The default run is a **pilot** whose audio comes **only** from the audio cache that
  Notebook 02 already created. The pilot **does not download Parquet shards** and
  **does not fetch any new audio**.

### ⚠️ What the pilot cache pool is (and is NOT)

The pilot samples `200 train / 50 validation` records **exclusively** from the
`notebook02_verified_cache_pool` — records whose cached WAV passes strict validation
(exists, mono, 16 kHz, finite, SHA-256 matches sidecar, `record_uid` matches, dataset
revision matches, audio processing version matches).

This pool exists **only to verify that the training pipeline runs end-to-end on Apple
MPS**. It is a *pipeline smoke test*. It is **not** a representative sample and its
metrics are **not** thesis results. A successful pilot does **not** mean full training is
done or that RQ1 is final.


In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
%pip install -q "jiwer>=3,<4"

In [ ]:
!python -m pytest -q

In [ ]:
# Cell 1 — Bootstrap (MPS CTC fallback MUST be set before importing torch)
import os
# Enable CPU fallback for MPS ops that lack a native kernel (e.g. some CTC bits).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import sys
from pathlib import Path

_NB_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
_PROJECT_ROOT = _NB_DIR.parent if _NB_DIR.name == "notebooks" else _NB_DIR
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

print(f"PYTORCH_ENABLE_MPS_FALLBACK={os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK')}")
print(f"Project root: {_PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
from src.asr_full_pcm import AUDIO_PCM_PIPELINE_VERSION
# Cell 2 — Imports
import io
import json
import time
import hashlib
import platform
import traceback
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.seed import set_seed
from src.data_utils import (
    normalize_bahnar_ctc_v1,
    encode_text_with_vocab,
    sha256_file,
    verify_tokenizer_provenance,
    build_clean_split,
    compute_ordered_uid_hash,
    compute_uid_set_hash,
    safe_cache_filename,
)
from src.asr_runtime_paths import (
    OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
    resolve_runtime_paths,
)
from src.asr_full_data import (
    PAIR_KEY_TRAIN_EXCLUSION_REASON,
    assert_eligible_frames_no_frozen_splits,
    compute_manifest_content_hash,
    drop_train_pair_key_overlaps,
    estimate_hydrate_wav_bytes,
    finalize_full_prepare,
    load_prepare_success,
    manifest_contract_fingerprint,
    vocab_fingerprint,
    write_effective_manifests,
)
from src.asr_full_train import (
    FULL_TRAIN_MARKER,
    RESUME_TEST_MARKER,
    RESUME_TEST_PHASE_A_STEPS,
    RESUME_TEST_PHASE_B_STEPS,
    assert_checkpoint_allowed_for_full_train,
    assert_checkpoint_complete_for_resume,
    assert_eligible_audio_available,
    assert_no_frozen_test_access,
    assert_ready_for_full_evaluate,
    assert_ready_for_full_train,
    build_full_training_hparams,
    build_data_contract,
    build_evaluate_contract,
    build_resume_test_contract,
    build_train_contract,
    build_training_contract,
    assert_training_contract,
    derive_full_evaluate_status,
    assert_durable_checkpoint_budget,
    durable_experiment_dir,
    durable_experiment_protected_bytes,
    estimate_checkpoint_bytes,
    resolve_durable_checkpoint_budget_bytes,
    ENV_DURABLE_CHECKPOINT_BUDGET_BYTES,
    measure_dir_bytes,
    plan_local_checkpoint_disk_peak,
    ensure_experiment_fingerprint,
    extract_training_contract,
    CANONICAL_METRIC_KEYS,
    canonicalize_trainer_metrics,
    metrics_are_finite,
    resolve_best_checkpoint_from_durable,
    training_contract_matches,
    build_resume_test_subset,
    build_validation_monitor_subset,
    collect_gpu_memory_snapshot,
    derive_full_resume_test_status,
    derive_full_training_status,
    experiment_checkpoint_dir,
    inspect_trainer_checkpoint,
    load_full_train_success,
    make_durable_checkpoint_sync_callback,
    resolve_eligible_audio_path,
    resolve_full_stage_status,
    resolve_resume_checkpoint,
    restore_experiment_checkpoints_from_durable,
    select_best_checkpoint_by_cer,
    sync_experiment_checkpoints_to_durable,
    write_checkpoint_fingerprint,
    write_full_train_summary,
    write_resume_test_summary,
)
from src.asr_utils import (

    AUDIO_EXCLUSIONS_COLUMNS,
    CTC_FEASIBILITY_EXCLUSIONS_COLUMNS,
    VALIDATION_PREDICTIONS_COLUMNS,
    OOV_SUMMARY_COLUMNS,
    EMPTY_NORMALIZED_REFERENCE,
    CTCDataCollatorWithPadding,
    analyze_oov,
    assert_full_mode_supported,
    build_eligible_pool_and_sample,
    check_cache_pool_sufficiency,
    check_ctc_feasibility,
    checkpoint_belongs_to_run,
    classify_cache_status,
    compute_batch_metrics,
    compute_cer,
    compute_records_and_hours,
    compute_wer,
    derive_pilot_status,
    detect_frozen_leakage,
    empty_dataframe,
    generate_run_id,
    get_device,
    get_environment_info,
    is_forbidden_test_path,
    preprocess_features_for_training,
    require_best_checkpoint,
    resolve_training_duration_seconds,
    sample_pilot_data,
    validate_prediction_lengths,
    verify_artifacts_run_id,
    verify_notebook03_prerequisites,
    verify_pilot_sample,
)
import transformers
import soundfile as sf

print(f"torch={torch.__version__}  transformers={transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available()}")


## Cấu hình thí nghiệm (pilot mặc định cho MacBook Pro M4 Pro / MPS)

In [ ]:
# Cell 3 — Configuration
# ---- Run mode -------------------------------------------------------------
RUN_MODE = "full"                  # "pilot" (default) or "full"
ALLOW_FULL_TRAINING = True         # must be explicitly True to attempt full mode
FULL_TRAINING_IMPLEMENTED = True   # prepare / resume_test / train / evaluate stages available
# Pilot NEVER downloads audio. It uses ONLY Notebook 02's verified audio cache.
ENABLE_AUDIO_DOWNLOAD = False

# ---- Full-training stages (only when RUN_MODE == "full") --------------------
# prepare -> resume_test_a -> resume_test_b -> train -> evaluate
FULL_STAGE = "prepare"  # prepare|resume_test_a|resume_test_b|resume_test|train|evaluate
# resume_test (legacy) runs A then B in one session; prefer A/B across RunPod pods
FULL_EXPERIMENT_ID = "full_xlsr300m_v1"
# ---- Runtime roots (RunPod) ------------------------------------------------
# LOCAL_ROOT: Parquet cache, WAV hydrate, temp, working checkpoints.
# DURABLE_ROOT: state, sidecar, contract, versioned checkpoints, metrics, export.
# Override with BAHNAR_LOCAL_ROOT / BAHNAR_DURABLE_ROOT.
RUNTIME_PATHS = resolve_runtime_paths(project_root=_PROJECT_ROOT)
LOCAL_ROOT = RUNTIME_PATHS.local_root
DURABLE_ROOT = RUNTIME_PATHS.durable_root
LOCAL_AUDIO_CACHE_DIR = RUNTIME_PATHS.audio_cache_dir
LOCAL_CKPT_ROOT = RUNTIME_PATHS.local_ckpt_root
HF_PARQUET_CACHE_DIR = RUNTIME_PATHS.hf_parquet_cache_dir
# Contract-scoped prepare/train state lives under DURABLE; assigned after the
# prepare contract is built (see F1). Placeholder until then.
FULL_STATE_DIR = RUNTIME_PATHS.durable_state_root
FULL_PARQUET_BATCH_SIZE = 32          # rows per pyarrow batch (never a whole shard in RAM)
FULL_SHARD_BYTES_HINT = 2 * 1024 ** 3  # fallback only; prefer measured shard sizes
LOCAL_DISK_RESERVE_BYTES = 5 * 1024 ** 3   # safety reserve on LOCAL_ROOT
LOCAL_HF_MODEL_CACHE_BYTES = 4 * 1024 ** 3  # xls-r-300m weights + processor artifacts
# Durable checkpoint budget (network volume / durable volume quota). Do NOT use
# shutil.disk_usage() on FUSE mounts as the source of truth for this budget.
DURABLE_CHECKPOINT_BUDGET_BYTES = None  # resolved in sync stages only via env
# xls-r-300m parameter count, used only for the pre-train disk estimate; the
# exact count is read from the model shell before training starts.
FULL_MODEL_PARAM_COUNT = 315_000_000
RESUME_POLICY = "auto"  # "auto" | "never" | "explicit"
FULL_RESUME_CHECKPOINT = None  # used only when RESUME_POLICY == "explicit"
FULL_NUM_TRAIN_EPOCHS = 1.0    # start with 1; max 3
FULL_SAVE_STEPS = 500
FULL_SAVE_TOTAL_LIMIT = 2
RESUME_TEST_SUBSET_SIZE = 64
FULL_VAL_MONITOR_SIZE = 64

# ---- Data contract (locked to Notebook 01/02) -----------------------------
DATASET_ID = "cuong06/Bahnar_Vietnamese"
EXPECTED_DATASET_REVISION = "3d88d3951b1a6e3388559b341cd7bd274879d696"
# Immutable snapshot of refs/convert/parquet. The alias itself is mutable, so it
# must never be used directly: audio bytes are bound to this sha in the contract.
EXPECTED_PARQUET_REVISION = "ad0a84362053dad098b86d0df215eb081f891dd8"
EXPECTED_TRAIN_CLEAN_COUNT = 102507
EXPECTED_VALIDATION_CLEAN_COUNT = 11113

# ---- Model ----------------------------------------------------------------
PRETRAINED_MODEL_ID = "facebook/wav2vec2-xls-r-300m"
PRETRAINED_MODEL_REVISION = "1a640f32ac3e39899438a2931f9924c02f080a54"

# ---- Audio ----------------------------------------------------------------
TARGET_SAMPLING_RATE = 16000
MIN_AUDIO_DURATION = 0.5
MAX_AUDIO_DURATION = 40.0
EXPECTED_AUDIO_PCM_PIPELINE_VERSION = AUDIO_PCM_PIPELINE_VERSION

# ---- Pilot configuration (first MPS pilot) --------------------------------
PILOT_TRAIN_SAMPLES = 200
PILOT_VALIDATION_SAMPLES = 50
MAX_STEPS = 20
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
PILOT_SEED = 42
SEED = 42

# Expected tokenizer facts (checked at load + at checkpoint reload).
EXPECTED_VOCAB_SIZE = 140
EXPECTED_PAD_ID = 0
EXPECTED_UNK_ID = 1
EXPECTED_DELIMITER_ID = 2

# ---- Optimizer ------------------------------------------------------------
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.0
WARMUP_STEPS = 5
EVAL_STEPS = MAX_STEPS          # single eval at the end for the tiny pilot
SAVE_STEPS = MAX_STEPS
LOGGING_STEPS = 1
MAX_CHECKPOINTS = 1

set_seed(SEED)

# ---- Device + MPS-aware knobs ---------------------------------------------
device = get_device(run_mode=RUN_MODE)
IS_CUDA = device.type == "cuda"
FP16 = IS_CUDA
DATALOADER_PIN_MEMORY = IS_CUDA   # pin_memory only helps on CUDA
GRADIENT_CHECKPOINTING = False if RUN_MODE == "pilot" else IS_CUDA

# ---- Run id + run-scoped artifact directories -----------------------------
RUN_ID = generate_run_id()

PATHS = {
    "manifests": _PROJECT_ROOT / "data" / "manifests",
    "audit": _PROJECT_ROOT / "data" / "audit",
    "cache": _PROJECT_ROOT / "data" / "cache",
    "tokenizer": _PROJECT_ROOT / "artifacts" / "tokenizers" / "bahnar_char_ctc",
    "contract": _PROJECT_ROOT / "results" / "notebook02_clean_split_contract.json",
    "summary_nb02": _PROJECT_ROOT / "results" / "notebook02_summary.json",
}
PATHS["artifacts_root"] = _PROJECT_ROOT / "artifacts" / "notebook03"
PATHS["results_root"] = _PROJECT_ROOT / "results" / "notebook03"
PATHS["checkpoints_root"] = _PROJECT_ROOT / "checkpoints" / "notebook03"
PATHS["artifacts"] = PATHS["artifacts_root"] / RUN_ID
PATHS["results"] = PATHS["results_root"] / RUN_ID
PATHS["checkpoints"] = PATHS["checkpoints_root"] / RUN_ID
for key in ["artifacts", "results", "checkpoints"]:
    PATHS[key].mkdir(parents=True, exist_ok=True)

AUDIO_CACHE_DIR = PATHS["cache"] / "notebook02_audio"

RESUME_FROM_CHECKPOINT = None

print(f"RUN_ID: {RUN_ID}")
print(f"RUN_MODE: {RUN_MODE}  device={device}  fp16={FP16}")
print(f"ENABLE_AUDIO_DOWNLOAD={ENABLE_AUDIO_DOWNLOAD} (pilot uses NB02 cache only)")
print(f"FULL_STAGE={FULL_STAGE}  FULL_EXPERIMENT_ID={FULL_EXPERIMENT_ID}")
print(f"LOCAL_ROOT={LOCAL_ROOT}")
print(f"DURABLE_ROOT={DURABLE_ROOT}  FULL_STATE_DIR={FULL_STATE_DIR}")
print(f"pin_memory={DATALOADER_PIN_MEMORY}  gradient_checkpointing={GRADIENT_CHECKPOINTING}")
print(f"pilot train/val={PILOT_TRAIN_SAMPLES}/{PILOT_VALIDATION_SAMPLES}  max_steps={MAX_STEPS}")
print(f"Artifacts dir: {PATHS['artifacts']}")


In [ ]:
# Cell 4 — Run-mode safety checks
assert_full_mode_supported(RUN_MODE, FULL_TRAINING_IMPLEMENTED)
if RUN_MODE == "full" and not ALLOW_FULL_TRAINING:
    raise RuntimeError(
        "RUN_MODE=full requires ALLOW_FULL_TRAINING=True "
        "(refusing accidental full-mode launch)."
    )
if RUN_MODE == "full":
    if FULL_STAGE not in {"prepare", "resume_test", "resume_test_a", "resume_test_b", "train", "evaluate"}:
        raise RuntimeError(f"Invalid FULL_STAGE={FULL_STAGE!r}")
    FULL_STATE_DIR = Path(FULL_STATE_DIR)
    LOCAL_ROOT = Path(LOCAL_ROOT)
    DURABLE_ROOT = Path(DURABLE_ROOT)
    LOCAL_AUDIO_CACHE_DIR = Path(LOCAL_AUDIO_CACHE_DIR)
    LOCAL_CKPT_ROOT = Path(LOCAL_CKPT_ROOT)
    print(
        f"Run-mode check PASSED (RUN_MODE=full, FULL_STAGE={FULL_STAGE}, "
        f"FULL_TRAINING_IMPLEMENTED={FULL_TRAINING_IMPLEMENTED})"
    )
else:
    print(
        f"Run-mode check PASSED (RUN_MODE={RUN_MODE}, "
        f"FULL_TRAINING_IMPLEMENTED={FULL_TRAINING_IMPLEMENTED})"
    )


## Frozen-test safety tracking (thật sự theo dõi, không hardcode)

In [ ]:
# Cell 5 — Frozen-test access guard
# Track every manifest/data path opened, every dataset split loaded, and every
# source split, then assert at the end that none aliases the frozen test set.
OPENED_PATHS: List[str] = []
LOADED_SPLITS: List[str] = []
DATA_SOURCES: List[str] = []

FROZEN_TEST_PATHS = {
    str((PATHS["manifests"] / "rq1_test.csv").resolve()),
    str((PATHS["manifests"] / "rq1_test.parquet").resolve()),
    str((PATHS["manifests"] / "rq1_test_candidate.csv").resolve()),
    str((PATHS["manifests"] / "rq1_test_candidate.parquet").resolve()),
}
FROZEN_TEST_SPLITS = {"test", "rq1_test", "frozen_test"}

def register_path(path) -> Path:
    p = Path(path)
    OPENED_PATHS.append(str(p.resolve()))
    if is_forbidden_test_path(p) or str(p.resolve()) in FROZEN_TEST_PATHS:
        raise RuntimeError(f"FROZEN TEST ACCESS BLOCKED: {p}")
    return p

def register_split(name: str) -> str:
    LOADED_SPLITS.append(str(name))
    if str(name).lower() in FROZEN_TEST_SPLITS:
        raise RuntimeError(f"FROZEN TEST SPLIT BLOCKED: {name}")
    return name

def register_source(name: str) -> str:
    DATA_SOURCES.append(str(name))
    return name

print("Frozen-test guard initialized.")


## Orchestration state (fail-safe: capture errors, always export artifacts)

Every heavy stage runs inside a guard. On failure we record `pipeline_error` +
`pipeline_error_stage`, set `can_continue=False`, and let downstream stages skip.
The artifact-export cells at the end **always run**, and only the very last cell
raises when `STATUS == "FAILED"`.

In [ ]:
# Cell 6 — Initialize pipeline state + ALL result variables (export must never NameError)
pipeline_error = None            # dict: exception_type/message/traceback
full_status = None               # SUCCESS_FULL_* or FAILED for full stages
full_summary = None
# Canonical contract of whichever full stage ran; without it a fallback to an
# older summary is never allowed.
full_stage_contract = None
pipeline_error_stage = None      # str
can_continue = True

def fail_stage(stage, exc):
    """Record a stage failure without raising; stop the pipeline gracefully."""
    global pipeline_error, pipeline_error_stage, can_continue
    if pipeline_error is None:
        pipeline_error = {
            "exception_type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        }
        pipeline_error_stage = stage
    can_continue = False
    print(f"[STAGE FAILED: {stage}] {type(exc).__name__}: {exc}")

# ---- contract / provenance ------------------------------------------------
prereq_check = {"passed": False, "errors": ["not_run"]}
tokenizer_prov = {"passed": False, "errors": ["not_run"]}
contract = {}
nb02_summary = {}
EXPECTED_AUDIO_PROCESSING_VERSION = None
EXPECTED_TRAIN_ORDERED_HASH = EXPECTED_TRAIN_SET_HASH = None
EXPECTED_VAL_ORDERED_HASH = EXPECTED_VAL_SET_HASH = None
train_ordered_hash = train_set_hash = val_ordered_hash = val_set_hash = None
hash_checks = []
data_contract_passed = False
tokenizer_provenance_passed = False

# ---- data -----------------------------------------------------------------
train_df = val_df = None
train_clean_df = val_clean_df = None
tokenizer = None
vocab = {}
VOCAB_CHARS = set()
feature_extractor = None
processor = None
model = None

oov_full_df = empty_dataframe(OOV_SUMMARY_COLUMNS)
oov_active_df = empty_dataframe(OOV_SUMMARY_COLUMNS)
oov_summary_df = empty_dataframe(OOV_SUMMARY_COLUMNS)

pilot_train_cache_pool = pd.DataFrame()
pilot_validation_cache_pool = pd.DataFrame()
cache_pool_summary = {}
cache_pool_sufficient = False

active_train_df = pd.DataFrame()
active_validation_df = pd.DataFrame()
empty_ref_exclusions = []          # rows for audio_exclusions.csv
ctc_exclusions = []                # rows for ctc_feasibility_exclusions.csv
ctc_feasibility_passed = False
active_counts_correct = False
overlaps_zero = False
source_split_ok = False

actual_train_records = 0
actual_validation_records = 0
actual_train_hours = 0.0
actual_validation_hours = 0.0

train_dataset = val_dataset = None
data_collator = None

# ---- training / checkpoint / eval -----------------------------------------
trainer = None
training_success = False
smoke_test_passed = False
train_result = None
training_start = None
training_duration = 0.0
train_runtime_seconds = None
perf_counter_seconds = None
duration_source = None
training_duration_source = None  # alias kept for older prints
train_samples_per_second = None
step_time = None
peak_memory_bytes = None

best_checkpoint_path = None
best_checkpoint_step = None
best_cer = None
best_metric = None
best_checkpoint_exists = False
checkpoint_reload_ok = False
checkpoint_stage_ok = False

eval_metrics = {}
predictions_df = empty_dataframe(VALIDATION_PREDICTIONS_COLUMNS)
predictions_count_match = False
validation_metrics_valid = False

frozen_test_accessed = False
all_integrity_passed = False
STATUS = "FAILED"

print("pipeline state + result variables initialized")


## Step 1 — Contract & provenance checks

In [ ]:
# Cell 7 — Prerequisites + Notebook 02 summary
if can_continue:
    try:
        print("=" * 60); print("CONTRACT VERIFICATION"); print("=" * 60)
        register_path(PATHS["contract"])
        prereq_check = verify_notebook03_prerequisites(
            contract_path=PATHS["contract"],
            tokenizer_dir=PATHS["tokenizer"],
            expected_dataset_revision=EXPECTED_DATASET_REVISION,
            expected_train_clean_count=EXPECTED_TRAIN_CLEAN_COUNT,
            expected_validation_clean_count=EXPECTED_VALIDATION_CLEAN_COUNT,
        )
        print(f"Prerequisite check: {'PASS' if prereq_check['passed'] else 'FAIL'}")
        if not prereq_check["passed"]:
            raise RuntimeError(f"Contract verification FAILED: {prereq_check['errors']}")

        register_path(PATHS["summary_nb02"])
        nb02_summary = json.loads(PATHS["summary_nb02"].read_text(encoding="utf-8"))
        assert nb02_summary.get("ready_for_notebook_03") is True, "NB02 not ready_for_notebook_03"
        EXPECTED_AUDIO_PROCESSING_VERSION = nb02_summary.get("processing_version")
        print(f"NB02 ready; expected audio processing version = {EXPECTED_AUDIO_PROCESSING_VERSION}")
    except Exception as _e:
        fail_stage("contract_verification", _e)


In [ ]:
# Cell 8 — Clean split contract hashes
if can_continue:
    try:
        contract = json.loads(PATHS["contract"].read_text(encoding="utf-8"))
        EXPECTED_TRAIN_ORDERED_HASH = contract["train"]["ordered_uid_sha256"]
        EXPECTED_TRAIN_SET_HASH = contract["train"]["uid_set_sha256"]
        EXPECTED_VAL_ORDERED_HASH = contract["validation"]["ordered_uid_sha256"]
        EXPECTED_VAL_SET_HASH = contract["validation"]["uid_set_sha256"]
        assert contract["dataset_revision"] == EXPECTED_DATASET_REVISION, "Contract revision mismatch"
        print(f"contract loaded: train clean={contract['train']['clean_count']:,} "
              f"val clean={contract['validation']['clean_count']:,}")
    except Exception as _e:
        fail_stage("contract_verification", _e)


In [ ]:
# Cell 9 — Tokenizer provenance verification
if can_continue:
    try:
        print("=" * 60); print("TOKENIZER PROVENANCE"); print("=" * 60)
        tokenizer_prov = verify_tokenizer_provenance(
            tokenizer_dir=PATHS["tokenizer"],
            expected_run_id=contract["run_id"],
            expected_revision=EXPECTED_DATASET_REVISION,
            expected_train_clean_ordered_uid_sha256=EXPECTED_TRAIN_ORDERED_HASH,
            expected_train_clean_uid_set_sha256=EXPECTED_TRAIN_SET_HASH,
        )
        print(f"Tokenizer provenance: {'PASS' if tokenizer_prov['passed'] else 'FAIL'}")
        if not tokenizer_prov["passed"]:
            raise RuntimeError(f"Tokenizer provenance FAILED: {tokenizer_prov['errors']}")
        tokenizer_provenance_passed = True
    except Exception as _e:
        fail_stage("tokenizer_provenance", _e)


## Step 2 — Load clean manifests (frozen test never loaded)

In [ ]:
# Cell 10 — Load clean manifests + build clean splits + hash/count/source_split checks
if can_continue:
    try:
        print("=" * 60); print("LOADING CLEAN MANIFESTS"); print("=" * 60)
        train_manifest_path = register_path(PATHS["manifests"] / "rq1_train.csv")
        val_manifest_path = register_path(PATHS["manifests"] / "rq1_validation.csv")
        register_split("train"); register_split("validation")
        register_source(f"{DATASET_ID}@{EXPECTED_DATASET_REVISION}")

        train_df = pd.read_csv(train_manifest_path)
        val_df = pd.read_csv(val_manifest_path)

        train_excl_path = register_path(PATHS["audit"] / "notebook02_train_contamination_exclusions.csv")
        val_excl_path = register_path(PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv")
        train_clean_df = build_clean_split(train_df, pd.read_csv(train_excl_path))
        val_clean_df = build_clean_split(val_df, pd.read_csv(val_excl_path))
        print(f"clean train={len(train_clean_df):,}  clean validation={len(val_clean_df):,}")

        assert len(train_clean_df) == EXPECTED_TRAIN_CLEAN_COUNT, "train clean count mismatch"
        assert len(val_clean_df) == EXPECTED_VALIDATION_CLEAN_COUNT, "validation clean count mismatch"

        # Frozen-test safety: BOTH clean splits must originate from the upstream 'train'
        # source split (the frozen RQ1 test comes from a different source split).
        assert set(train_clean_df["source_split"].astype(str)) == {"train"}, "train source_split != {train}"
        assert set(val_clean_df["source_split"].astype(str)) == {"train"}, "val source_split != {train}"

        # UID hash verification against the locked contract.
        train_ordered_hash = compute_ordered_uid_hash(train_clean_df)
        train_set_hash = compute_uid_set_hash(train_clean_df)
        val_ordered_hash = compute_ordered_uid_hash(val_clean_df)
        val_set_hash = compute_uid_set_hash(val_clean_df)
        hash_checks = [
            ("train_ordered", train_ordered_hash == EXPECTED_TRAIN_ORDERED_HASH),
            ("train_set", train_set_hash == EXPECTED_TRAIN_SET_HASH),
            ("validation_ordered", val_ordered_hash == EXPECTED_VAL_ORDERED_HASH),
            ("validation_set", val_set_hash == EXPECTED_VAL_SET_HASH),
        ]
        if not all(ok for _, ok in hash_checks):
            raise RuntimeError(f"UID hash verification FAILED: {hash_checks}")

        REQUIRED_COLUMNS = [
            "record_uid", "record_id", "group_id", "audio_path", "text_bahnar",
            "duration_seconds", "parquet_file", "shard_row_index",
            "source_split", "recording_group_id", "pair_key", "split",
        ]
        for col in REQUIRED_COLUMNS:
            assert col in train_clean_df.columns, f"missing train column: {col}"
            assert col in val_clean_df.columns, f"missing validation column: {col}"

        data_contract_passed = bool(
            prereq_check["passed"] and all(ok for _, ok in hash_checks)
            and len(train_clean_df) == EXPECTED_TRAIN_CLEAN_COUNT
            and len(val_clean_df) == EXPECTED_VALIDATION_CLEAN_COUNT
        )
        print(f"count+hash+source_split+columns verified; data_contract_passed={data_contract_passed}")
    except Exception as _e:
        fail_stage("manifest_load", _e)


## Step 3 — Tokenizer, processor, model

In [ ]:
# Cell 11 — Load tokenizer, feature extractor, processor, model
if can_continue:
    try:
        print("=" * 60); print("LOADING TOKENIZER / MODEL"); print("=" * 60)
        from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                                  Wav2Vec2Processor, Wav2Vec2ForCTC)
        tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(str(PATHS["tokenizer"]))
        vocab = json.loads((PATHS["tokenizer"] / "vocab.json").read_text(encoding="utf-8"))
        VOCAB_CHARS = set(vocab.keys())
        assert len(vocab) == EXPECTED_VOCAB_SIZE, f"vocab size {len(vocab)} != {EXPECTED_VOCAB_SIZE}"
        assert tokenizer.pad_token_id == EXPECTED_PAD_ID, "PAD id mismatch"
        assert tokenizer.unk_token_id == EXPECTED_UNK_ID, "UNK id mismatch"
        assert tokenizer.word_delimiter_token_id == EXPECTED_DELIMITER_ID, "delimiter id mismatch"

        feature_extractor = Wav2Vec2FeatureExtractor(
            feature_size=1, sampling_rate=TARGET_SAMPLING_RATE, padding_value=0.0,
            do_normalize=True, return_attention_mask=True,
        )
        assert feature_extractor.sampling_rate == TARGET_SAMPLING_RATE
        processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

        # Full mode: never preload base model here (stages load shell/checkpoint themselves).
        # Pilot still loads base model for training in this cell.
        if RUN_MODE == "full":
            model = None
            print(f"tokenizer/vocab OK: vocab={len(tokenizer)} (full mode: no Cell-11 base preload)")
        else:
            model = Wav2Vec2ForCTC.from_pretrained(
                PRETRAINED_MODEL_ID, revision=PRETRAINED_MODEL_REVISION,
                ctc_loss_reduction="mean", pad_token_id=tokenizer.pad_token_id,
                vocab_size=len(tokenizer), ignore_mismatched_sizes=True,
            )
            model.freeze_feature_encoder()
            (model.gradient_checkpointing_enable() if GRADIENT_CHECKPOINTING
             else model.gradient_checkpointing_disable())
            assert model.config.vocab_size == len(tokenizer), "model vocab mismatch"
            model.to(device)
            print(
                f"tokenizer/model OK: vocab={len(tokenizer)} pad={tokenizer.pad_token_id} "
                f"grad_ckpt={GRADIENT_CHECKPOINTING}"
            )
    except Exception as _e:
        fail_stage("model_load", _e)


## Step 4 — OOV on FULL clean validation (do NOT add val-only chars to vocab)

In [ ]:
# Cell 12 — OOV on full clean validation
if can_continue:
    try:
        # analyze_oov normalizes with normalize_bahnar_ctc_v1 before counting.
        oov_full_df = analyze_oov(
            val_clean_df["text_bahnar"].tolist(),
            vocab,
            scope="full_clean_validation",
            normalize_fn=normalize_bahnar_ctc_v1,
        )
        print(f"Full clean validation OOV chars: {len(oov_full_df)} (mapped to [UNK]; vocab NOT modified)")
    except Exception as _e:
        fail_stage("oov_full", _e)


## Step 5 — Build the pilot verified cache pool (NO Parquet, NO download)

We scan Notebook 02's audio cache and keep only records whose cached WAV passes
strict validation. The pilot samples exclusively from this pool.

In [ ]:
# Cell 13 — Build pilot_train_cache_pool / pilot_validation_cache_pool (strict validation only)
if RUN_MODE == "pilot":
    if can_continue:
        try:
            assert RUN_MODE == "pilot", "This cell builds the PILOT cache pool only."
            assert ENABLE_AUDIO_DOWNLOAD is False, "Pilot must not download audio."
            AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
            cached_files = {p.name for p in AUDIO_CACHE_DIR.glob("*.wav")}
            print(f"cached WAVs available on disk: {len(cached_files)}")
    
            def build_verified_cache_pool(clean_df, split):
                valid_uids, dur_map, reasons = [], {}, {}
                for _, r in clean_df.iterrows():
                    uid = str(r["record_uid"])
                    fname = safe_cache_filename(uid)
                    if fname not in cached_files:
                        continue  # not cached by NB02 -> simply not in the pool
                    st = classify_cache_status(
                        AUDIO_CACHE_DIR / fname, record_uid=uid,
                        dataset_revision=EXPECTED_DATASET_REVISION,
                        target_sr=TARGET_SAMPLING_RATE, min_duration=MIN_AUDIO_DURATION,
                        max_duration=MAX_AUDIO_DURATION,
                        expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION)
                    if st["ok"]:
                        valid_uids.append(uid); dur_map[uid] = st["duration_seconds"]
                    else:
                        reasons[st["reason"]] = reasons.get(st["reason"], 0) + 1
                pool = clean_df[clean_df["record_uid"].astype(str).isin(set(valid_uids))].copy()
                pool["processed_duration_seconds"] = pool["record_uid"].astype(str).map(dur_map)
                print(f"[{split}] cache pool: strict_valid={len(pool)}  reject_reasons={reasons}")
                return pool, reasons
    
            pilot_train_cache_pool, train_pool_reasons = build_verified_cache_pool(train_clean_df, "train")
            pilot_validation_cache_pool, val_pool_reasons = build_verified_cache_pool(val_clean_df, "validation")
    
            train_suff = check_cache_pool_sufficiency(len(pilot_train_cache_pool), PILOT_TRAIN_SAMPLES, "train")
            val_suff = check_cache_pool_sufficiency(len(pilot_validation_cache_pool), PILOT_VALIDATION_SAMPLES, "validation")
            cache_pool_summary = {
                "run_id": RUN_ID,
                "expected_audio_processing_version": EXPECTED_AUDIO_PROCESSING_VERSION,
                "train": {**train_suff, "reject_reasons": train_pool_reasons},
                "validation": {**val_suff, "reject_reasons": val_pool_reasons},
            }
            cache_pool_sufficient = bool(train_suff["ok"] and val_suff["ok"])
            print(train_suff["message"]); print(val_suff["message"])
            if not cache_pool_sufficient:
                raise RuntimeError(
                    "Verified cache pool is insufficient for the pilot (NO Parquet backfill). "
                    f"train: {train_suff['message']} | validation: {val_suff['message']}"
                )
        except Exception as _e:
            fail_stage("cache_pool", _e)


## Step 6 — Select 200/50 (group-aware) with empty-reference + CTC filtering

Selection draws from the verified pool in a deterministic, group-aware order.
Records with an empty normalized transcript or that are CTC-infeasible are
excluded and **deterministically replaced** by the next pool record (no download).

In [ ]:
# Cell 14 — Select active train/validation: filter eligible pool FIRST, then group-aware sample
# FIX: The old code called sample_pilot_data(pool, len(pool)) which returns pool in original
# order (sorted by group+uid, NOT shuffled). This meant we took the first N valid records
# instead of doing true group-aware selection.
#
# New approach:
# 1. Filter ENTIRE pool for empty refs + CTC feasibility → eligible_pool
# 2. Check eligible_pool >= target_n (raise if not)
# 3. Call sample_pilot_data(eligible_pool, target_n, ...) for actual group-aware selection

# Initialize group coverage metadata (used in cell 15)
if RUN_MODE == "pilot":
    train_group_meta = {}
    validation_group_meta = {}
    
    if can_continue:
        try:
            # Train selection
            train_result = build_eligible_pool_and_sample(
                pool_df=pilot_train_cache_pool,
                target_n=PILOT_TRAIN_SAMPLES,
                split="train",
                seed=PILOT_SEED,
                vocab=vocab,
                normalize_fn=normalize_bahnar_ctc_v1,
                ctc_check_fn=check_ctc_feasibility,
                encode_fn=encode_text_with_vocab,
                target_sr=TARGET_SAMPLING_RATE,
                run_id=RUN_ID,
            )
            active_train_df = train_result["active_df"]
            tr_empty = train_result["empty_ref_exclusions"]
            tr_ctc = train_result["ctc_exclusions"]
            train_group_meta = {
                "eligible_pool_size": train_result["eligible_pool_size"],
                "eligible_group_count": train_result["eligible_group_count"],
                "active_group_count": train_result["active_group_count"],
                "expected_active_groups": train_result["expected_active_groups"],
                "group_coverage_ratio": train_result["group_coverage_ratio"],
            }
    
            # Validation selection
            validation_result = build_eligible_pool_and_sample(
                pool_df=pilot_validation_cache_pool,
                target_n=PILOT_VALIDATION_SAMPLES,
                split="validation",
                seed=PILOT_SEED,
                vocab=vocab,
                normalize_fn=normalize_bahnar_ctc_v1,
                ctc_check_fn=check_ctc_feasibility,
                encode_fn=encode_text_with_vocab,
                target_sr=TARGET_SAMPLING_RATE,
                run_id=RUN_ID,
            )
            active_validation_df = validation_result["active_df"]
            va_empty = validation_result["empty_ref_exclusions"]
            va_ctc = validation_result["ctc_exclusions"]
            validation_group_meta = {
                "eligible_pool_size": validation_result["eligible_pool_size"],
                "eligible_group_count": validation_result["eligible_group_count"],
                "active_group_count": validation_result["active_group_count"],
                "expected_active_groups": validation_result["expected_active_groups"],
                "group_coverage_ratio": validation_result["group_coverage_ratio"],
            }
    
            empty_ref_exclusions = tr_empty + va_empty
            ctc_exclusions = tr_ctc + va_ctc
            ctc_feasibility_passed = True
    
            print(f"TRAIN: eligible records={train_group_meta['eligible_pool_size']}, "
                  f"eligible groups={train_group_meta['eligible_group_count']}, "
                  f"selected records={len(active_train_df)}, "
                  f"selected groups={train_group_meta['active_group_count']}")
            print(f"VALIDATION: eligible records={validation_group_meta['eligible_pool_size']}, "
                  f"eligible groups={validation_group_meta['eligible_group_count']}, "
                  f"selected records={len(active_validation_df)}, "
                  f"selected groups={validation_group_meta['active_group_count']}")
            print(f"empty_excl={len(empty_ref_exclusions)} ctc_excl={len(ctc_exclusions)}")
        except Exception as _e:
            fail_stage("data_preparation", _e)


In [ ]:
# Cell 15 — Recompute counts/hours, overlap + source_split checks, export manifests + metadata
if RUN_MODE == "pilot":
    if can_continue:
        try:
            tr = compute_records_and_hours(active_train_df)
            va = compute_records_and_hours(active_validation_df)
            actual_train_records, actual_train_hours = tr["records"], tr["hours"]
            actual_validation_records, actual_validation_hours = va["records"], va["hours"]
    
            active_counts_correct = bool(
                actual_train_records == PILOT_TRAIN_SAMPLES
                and actual_validation_records == PILOT_VALIDATION_SAMPLES)
    
            def _overlap(col):
                if col not in active_train_df.columns or col not in active_validation_df.columns:
                    return None
                s = set(active_train_df[col].dropna().astype(str)) & set(active_validation_df[col].dropna().astype(str))
                return len(s)
            overlaps = {c: _overlap(c) for c in ["record_uid", "group_id", "recording_group_id", "pair_key"]}
            overlaps_zero = all((v == 0) for v in overlaps.values() if v is not None)
            source_split_ok = (set(active_train_df["source_split"].astype(str)) == {"train"}
                               and set(active_validation_df["source_split"].astype(str)) == {"train"})
            print(f"counts_correct={active_counts_correct} overlaps={overlaps} "
                  f"overlaps_zero={overlaps_zero} source_split_ok={source_split_ok}")
            if not (active_counts_correct and overlaps_zero and source_split_ok):
                raise RuntimeError(f"Post-selection integrity failed: counts={active_counts_correct} "
                                   f"overlaps={overlaps} source_split_ok={source_split_ok}")
    
            # Pilot manifests reflect the final active selection.
            active_train_df.to_csv(PATHS["artifacts"] / "pilot_train_manifest.csv", index=False)
            active_validation_df.to_csv(PATHS["artifacts"] / "pilot_validation_manifest.csv", index=False)
            active_train_df.to_csv(PATHS["artifacts"] / "active_train_manifest.csv", index=False)
            active_validation_df.to_csv(PATHS["artifacts"] / "active_validation_manifest.csv", index=False)
    
            pilot_cache_metadata = {
                "run_id": RUN_ID,
                "pilot_sampling_scope": "notebook02_verified_cache_pool",
                "pilot_is_representative_final_evaluation": False,
                "pilot_purpose": "pipeline_smoke_test_only",
                "train_cache_pool_size": int(len(pilot_train_cache_pool)),
                "validation_cache_pool_size": int(len(pilot_validation_cache_pool)),
                "pilot_train_target": PILOT_TRAIN_SAMPLES,
                "pilot_validation_target": PILOT_VALIDATION_SAMPLES,
                "actual_train_records": actual_train_records,
                "actual_validation_records": actual_validation_records,
                "split_overlaps": overlaps,
                # Group coverage metrics (fix for group-aware sampling)
                "train_group_coverage": train_group_meta if train_group_meta else None,
                "validation_group_coverage": validation_group_meta if validation_group_meta else None,
            }
            (PATHS["artifacts"] / "pilot_cache_metadata.json").write_text(
                json.dumps(pilot_cache_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"train_hours={actual_train_hours:.3f} val_hours={actual_validation_hours:.3f}")
        except Exception as _e:
            fail_stage("data_preparation", _e)


In [ ]:
# Cell 16 — Active OOV + combined OOV export
if RUN_MODE == "pilot":
    if can_continue:
        try:
            # analyze_oov normalizes with normalize_bahnar_ctc_v1 before counting.
            oov_active_df = analyze_oov(
                active_validation_df["text_bahnar"].tolist(),
                vocab,
                scope="active_validation",
                normalize_fn=normalize_bahnar_ctc_v1,
            )
            oov_summary_df = pd.concat([oov_full_df, oov_active_df], ignore_index=True) \
                if (len(oov_full_df) or len(oov_active_df)) else empty_dataframe(OOV_SUMMARY_COLUMNS)
            print(f"OOV: full_clean={len(oov_full_df)} active={len(oov_active_df)}")
        except Exception as _e:
            fail_stage("oov_active", _e)


## Step 7 — Dataset & collator (single feature-extractor preprocessing path)

In [ ]:
# Cell 17 — ASR dataset + collator
if RUN_MODE == "pilot":
    if can_continue:
        try:
            from torch.utils.data import Dataset
    
            class ASRDataset(Dataset):
                def __init__(self, df, vocab, cache_dir, processor, target_sr=TARGET_SAMPLING_RATE):
                    self.df = df.reset_index(drop=True)
                    self.vocab = vocab
                    self.cache_dir = Path(cache_dir)
                    self.processor = processor
                    self.target_sr = target_sr
                    self.df["text_norm"] = self.df["text_bahnar"].apply(normalize_bahnar_ctc_v1)
    
                def __len__(self):
                    return len(self.df)
    
                def __getitem__(self, idx):
                    row = self.df.iloc[idx]
                    uid = str(row["record_uid"])
                    wav, sr = sf.read(self.cache_dir / safe_cache_filename(uid), dtype="float32")
                    input_values = preprocess_features_for_training(self.processor, wav, sr, self.target_sr)
                    labels = encode_text_with_vocab(row["text_bahnar"], self.vocab)
                    return {"input_values": input_values, "labels": labels, "record_uid": uid,
                            "record_id": row.get("record_id"), "group_id": row.get("group_id"),
                            "duration_seconds": float(row.get("processed_duration_seconds") or 0.0),
                            "reference_raw": row["text_bahnar"], "reference_normalized": row["text_norm"]}
    
            train_dataset = ASRDataset(active_train_df, vocab, AUDIO_CACHE_DIR, processor)
            val_dataset = ASRDataset(active_validation_df, vocab, AUDIO_CACHE_DIR, processor)
            data_collator = CTCDataCollatorWithPadding(processor=processor)
    
            _probe = data_collator([train_dataset[i] for i in range(min(2, len(train_dataset)))])
            assert torch.isfinite(_probe["input_values"]).all(), "non-finite input_values in sanity batch"
            print(f"train_dataset={len(train_dataset)} val_dataset={len(val_dataset)} sanity batch OK")
        except Exception as _e:
            fail_stage("dataset_build", _e)


## Step 8 — Trainer, gradient smoke test, training

In [ ]:
# Cell 18 — TrainingArguments + compute_metrics + Trainer
if RUN_MODE == "pilot":
    if can_continue:
        try:
            from transformers import Trainer, TrainingArguments
            training_args = TrainingArguments(
                output_dir=str(PATHS["checkpoints"]),
                run_name=f"bahnar_asr_{RUN_MODE}_{RUN_ID[:8]}",
                max_steps=MAX_STEPS, num_train_epochs=1,
                per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
                per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
                gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, warmup_steps=WARMUP_STEPS,
                eval_strategy="steps", eval_steps=EVAL_STEPS,
                save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=MAX_CHECKPOINTS,
                load_best_model_at_end=True, metric_for_best_model="cer", greater_is_better=False,
                fp16=FP16, dataloader_num_workers=0, dataloader_pin_memory=DATALOADER_PIN_MEMORY,
                gradient_checkpointing=GRADIENT_CHECKPOINTING, remove_unused_columns=False,
                # logging_dir removed: unsupported on transformers>=5 (use TENSORBOARD_LOGGING_DIR if needed)
                logging_steps=LOGGING_STEPS,
                report_to=[], seed=SEED, data_seed=SEED,
            )
    
            def compute_metrics(eval_pred):
                logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
                labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]
                if isinstance(logits, (tuple, list)):
                    logits = logits[0]
                pred_ids = np.argmax(logits, axis=-1)
                labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
                refs = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
                hyps = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
                # Empty references are excluded upstream -> enforce it here too.
                m = compute_batch_metrics(refs, hyps, allow_empty_reference=False)
                return {"cer": m["cer"], "wer": m["wer"]}
    
            try:
                trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                                  eval_dataset=val_dataset, data_collator=data_collator,
                                  compute_metrics=compute_metrics, processing_class=processor)
            except TypeError:
                trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                                  eval_dataset=val_dataset, data_collator=data_collator,
                                  compute_metrics=compute_metrics, tokenizer=processor)
            print(f"Trainer ready: {type(trainer).__name__}")
        except Exception as _e:
            fail_stage("trainer_build", _e)


In [ ]:
# Cell 19 — Gradient smoke test BEFORE trainer.train() (captures error, does NOT raise)
if RUN_MODE == "pilot":
    if can_continue:
        try:
            from torch.utils.data import DataLoader
            print("=" * 60); print("GRADIENT SMOKE TEST"); print("=" * 60)
            model.train(); model.to(device)
            _loader = DataLoader(train_dataset, batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
                                 collate_fn=data_collator, shuffle=False)
            _batch = next(iter(_loader))
            _batch = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in _batch.items()}
            _out = model(**_batch); _loss = _out.loss
            if _loss is None or not torch.isfinite(_loss).all():
                raise RuntimeError(f"Non-finite/None loss in smoke test: {_loss}")
            _loss.backward()
            _grad_ok = any(
                (p.requires_grad and p.grad is not None and torch.isfinite(p.grad).all()
                 and float(p.grad.abs().sum()) > 0.0)
                for p in model.parameters())
            model.zero_grad(set_to_none=True)
            smoke_test_passed = bool(_grad_ok)
            print(f"smoke loss={float(_loss):.4f}  finite_nonzero_grad={_grad_ok}")
            if not smoke_test_passed:
                raise RuntimeError("No finite non-zero gradient found on any trainable parameter")
        except Exception as _e:
            fail_stage("gradient_smoke_test", _e)


In [ ]:
# Cell 20 — Run training (captures error, does NOT raise)
if RUN_MODE == "pilot":
    if can_continue:
        try:
            print("=" * 60); print(f"TRAINING ({RUN_MODE.upper()})  max_steps={MAX_STEPS}"); print("=" * 60)
            print("PILOT: pipeline validation only, NOT thesis results.")
            training_start = datetime.now(timezone.utc)
            _t0 = time.perf_counter()
            train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
            perf_counter_seconds = time.perf_counter() - _t0
            training_success = True
            tm = train_result.metrics or {}
            train_samples_per_second = tm.get("train_samples_per_second")
            _dur = resolve_training_duration_seconds(
                tm.get("train_runtime"),
                perf_counter_seconds,
                global_step=getattr(train_result, "global_step", 1),
            )
            # Official duration prefers train_runtime; perf_counter is diagnostic/fallback only.
            training_duration = _dur["training_duration_seconds"]
            train_runtime_seconds = _dur["train_runtime_seconds"]
            perf_counter_seconds = _dur["perf_counter_seconds"]
            duration_source = _dur["duration_source"]
            training_duration_source = duration_source
            step_time = _dur["step_time_seconds"]
            try:
                if device.type == "cuda":
                    peak_memory_bytes = int(torch.cuda.max_memory_allocated())
                elif device.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "current_allocated_memory"):
                    peak_memory_bytes = int(torch.mps.current_allocated_memory())
            except Exception:
                peak_memory_bytes = None
            print(
                f"training done in {training_duration:.1f}s "
                f"(source={duration_source}, train_runtime={train_runtime_seconds}, "
                f"perf_counter={perf_counter_seconds}) "
                f"global_step={train_result.global_step} step_time={step_time:.3f}s"
            )
        except Exception as _e:
            _pc = time.perf_counter() - _t0 if "_t0" in dir() else 0.0
            _dur = resolve_training_duration_seconds(None, _pc, global_step=1)
            training_duration = _dur["training_duration_seconds"]
            train_runtime_seconds = _dur["train_runtime_seconds"]
            perf_counter_seconds = _dur["perf_counter_seconds"]
            duration_source = _dur["duration_source"]
            training_duration_source = duration_source
            step_time = _dur["step_time_seconds"]
            fail_stage("training", _e)


## Step 9 — Best checkpoint (run-scoped, no fallback) + validation via trainer.predict

In [ ]:
# Cell 21 — Resolve best checkpoint of THIS run, verify processor/vocab, reload model
if RUN_MODE == "pilot":
    if can_continue:
        try:
            from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
            # STRICT: only trainer.state.best_model_checkpoint; NO highest-step fallback.
            best_checkpoint_path = require_best_checkpoint(getattr(trainer.state, "best_model_checkpoint", None))
            best_metric = getattr(trainer.state, "best_metric", None)
            best_cer = best_metric
            if not checkpoint_belongs_to_run(best_checkpoint_path, PATHS["checkpoints"]):
                raise RuntimeError(f"best checkpoint {best_checkpoint_path} is not inside "
                                   f"this run's checkpoint dir {PATHS['checkpoints']}")
            best_checkpoint_exists = True
            bcp = Path(best_checkpoint_path)
            try:
                best_checkpoint_step = int(bcp.name.split("-")[-1])
            except Exception:
                best_checkpoint_step = None
    
            processor.save_pretrained(str(bcp))
            reloaded = Wav2Vec2ForCTC.from_pretrained(str(bcp))
            reloaded.eval()
            with torch.no_grad():
                _ = reloaded(torch.randn(1, TARGET_SAMPLING_RATE))
            rp = Wav2Vec2Processor.from_pretrained(str(bcp))
            rtok = rp.tokenizer
            assert len(rtok) == EXPECTED_VOCAB_SIZE, "reloaded vocab size mismatch"
            assert rtok.pad_token_id == EXPECTED_PAD_ID, "reloaded PAD id mismatch"
            assert rtok.unk_token_id == EXPECTED_UNK_ID, "reloaded UNK id mismatch"
            _rv = json.loads((bcp / "vocab.json").read_text(encoding="utf-8"))
            assert _rv.get("|") == EXPECTED_DELIMITER_ID, "reloaded delimiter id mismatch"
            checkpoint_reload_ok = True
            checkpoint_stage_ok = True
            del reloaded
            print(f"best checkpoint OK: {bcp.name} step={best_checkpoint_step} reload+processor verified")
        except Exception as _e:
            checkpoint_stage_ok = False
            fail_stage("checkpoint", _e)


In [ ]:
# Cell 22 — Validation via a single trainer.predict (strict length check, no truncation)
if RUN_MODE == "pilot":
    if can_continue:
        try:
            val_meta = active_validation_df.reset_index(drop=True)
            pred_out = trainer.predict(val_dataset)
            eval_metrics = dict(pred_out.metrics or {})
            logits = pred_out.predictions
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            label_ids = np.where(pred_out.label_ids != -100, pred_out.label_ids, tokenizer.pad_token_id)
            pred_ids = np.argmax(logits, axis=-1)
            pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
            # NO silent truncation: predictions/labels/metadata/dataset must all match.
            validate_prediction_lengths(len(pred_str), len(label_str), len(val_meta), len(val_dataset))
    
            ckpt_name = Path(best_checkpoint_path).name if best_checkpoint_path else "final"
            refs, hyps, rows = [], [], []
            for i in range(len(val_meta)):
                r = val_meta.iloc[i]
                ref_norm = normalize_bahnar_ctc_v1(r["text_bahnar"])
                pred_norm = normalize_bahnar_ctc_v1(pred_str[i])
                refs.append(ref_norm); hyps.append(pred_norm)
                rows.append({"run_id": RUN_ID, "checkpoint": ckpt_name,
                             "record_uid": str(r["record_uid"]), "record_id": r.get("record_id"),
                             "group_id": r.get("group_id"),
                             "duration_seconds": float(r.get("processed_duration_seconds") or 0.0),
                             "reference_raw": r["text_bahnar"], "prediction_raw": pred_str[i],
                             "reference_normalized": ref_norm, "prediction_normalized": pred_norm,
                             "cer": compute_cer(ref_norm, pred_norm),
                             "wer": compute_wer(ref_norm, pred_norm)})
            predictions_df = pd.DataFrame(rows, columns=VALIDATION_PREDICTIONS_COLUMNS)
            # Empty references were excluded upstream -> allow_empty_reference=False here.
            corpus = compute_batch_metrics(refs, hyps, allow_empty_reference=False)
            eval_metrics.setdefault("corpus_cer", corpus["cer"])
            eval_metrics.setdefault("corpus_wer", corpus["wer"])
            predictions_count_match = (len(predictions_df) == len(active_validation_df))
            validation_metrics_valid = bool(predictions_count_match and np.isfinite(corpus["cer"]))
            print(f"predictions={len(predictions_df)} count_match={predictions_count_match} "
                  f"corpus_cer={corpus['cer']:.4f} corpus_wer={corpus['wer']:.4f}")
        except Exception as _e:
            fail_stage("evaluation", _e)


## Full-mode stages (RUN_MODE="full")

Stages are explicit and durable under `FULL_STATE_DIR` (DURABLE_ROOT on the network volume).
Pilot path above is skipped when `RUN_MODE != "pilot"`.

| FULL_STAGE | Meaning |
|---|---|
| `prepare` | Shard-sequential Parquet prepare → eligible CSVs (no 100k WAVs on durable storage) |
| `resume_test` | Separate experiment: train to 100, resume to 200 |
| `train` | Full training from locked XLS-R base (never pilot/resume-test ckpt) |
| `evaluate` | Reload best full checkpoint; full eligible validation |


In [ ]:
# Cell F1 — FULL_STAGE=prepare (one pass over parquet shards, resumable)
full_status = None
full_summary = None
if can_continue and RUN_MODE == "full" and FULL_STAGE == "prepare":
    try:
        print("=" * 60); print("FULL PREPARE"); print("=" * 60)
        LOCAL_AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        HF_PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

        # Effective manifests (Notebooks 03–05): drop train rows whose pair_key
        # appears in validation BEFORE any audio decode. Validation unchanged.
        train_effective, pair_key_excl, pair_key_report = drop_train_pair_key_overlaps(
            train_clean_df, val_clean_df,
        )
        val_effective = val_clean_df.copy()
        print(json.dumps(pair_key_report, indent=2))
        eff_dir = RUNTIME_PATHS.durable_state_root / "effective_manifests"
        eff_summary = write_effective_manifests(
            state_dir=eff_dir,
            train_df=train_effective,
            val_df=val_effective,
            train_exclusions=pair_key_excl,
            source_train_path=str(PATHS["manifests"] / "rq1_train.csv"),
            source_val_path=str(PATHS["manifests"] / "rq1_validation.csv"),
        )
        # Bind prepare to effective manifest hashes + MAX=40 + overlap policy.
        prepare_contract_early = build_data_contract(
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=eff_summary["train_manifest_content_hash"],
            validation_manifest_content_hash=eff_summary["validation_manifest_content_hash"],
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
        )
        FULL_STATE_DIR = RUNTIME_PATHS.prepare_state_dir(prepare_contract_early["contract_hash"])
        FULL_STATE_DIR.mkdir(parents=True, exist_ok=True)
        # Mirror effective manifests into the contract-scoped state dir.
        write_effective_manifests(
            state_dir=FULL_STATE_DIR,
            train_df=train_effective,
            val_df=val_effective,
            train_exclusions=pair_key_excl,
            source_train_path=str(PATHS["manifests"] / "rq1_train.csv"),
            source_val_path=str(PATHS["manifests"] / "rq1_validation.csv"),
        )
        print(f"FULL_STATE_DIR (contract-scoped)={FULL_STATE_DIR}")

        from src.asr_full_data import (
            assert_local_disk_budget,
            assert_parquet_files_are_train_only,
            build_shard_plans,
            cleanup_shard_download,
            fetch_parquet_shard_sizes,
            drop_train_pair_key_overlaps,
            write_effective_manifests,
            estimate_hydrate_wav_bytes,
            make_hf_parquet_stream_reader,
            run_full_prepare_streaming,
            verify_pinned_parquet_snapshot,
        )

        # Manifests must only ever point at the upstream train shards; the
        # snapshot also carries default/test/* and default/validation/*.
        shard_files = assert_parquet_files_are_train_only(
            [
                p.ref.filename
                for p in build_shard_plans(
                    {"train": train_effective, "validation": val_effective},
                    dataset_id=DATASET_ID,
                    parquet_revision=EXPECTED_PARQUET_REVISION,
                )
            ]
        )
        # Read-only preflight: the pinned commit must resolve and hold every
        # shard we need. A moved refs/convert/parquet is only a warning.
        pq_preflight = verify_pinned_parquet_snapshot(
            repo_id=DATASET_ID,
            pinned_revision=EXPECTED_PARQUET_REVISION,
            required_filenames=shard_files,
        )
        for _w in pq_preflight["warnings"]:
            print(f"  WARNING parquet snapshot: {_w}")
        print(
            f"parquet snapshot {pq_preflight['pinned_revision'][:12]} verified: "
            f"{pq_preflight['required_count']} shards required"
        )

        # Budget from the shards we will actually download, not a constant: a
        # guessed size makes the preflight decorative. Fail-closed, because
        # running out of disk mid-shard is what we are trying to prevent.
        shard_sizes = fetch_parquet_shard_sizes(
            repo_id=DATASET_ID,
            revision=EXPECTED_PARQUET_REVISION,
            filenames=shard_files,
        )
        largest_shard_bytes = max(shard_sizes.values())
        print(
            f"largest required shard: {largest_shard_bytes / 1e9:.2f}GB "
            f"(total {sum(shard_sizes.values()) / 1e9:.1f}GB across {len(shard_sizes)})"
        )

        # Prepare is metadata-only, so the only local need is one parquet shard
        # at a time plus temporary download space.
        print(json.dumps(assert_local_disk_budget(
            HF_PARQUET_CACHE_DIR,
            needs={
                "largest_parquet_shard": largest_shard_bytes,
                "download_temporary": largest_shard_bytes,
            },
            reserve_bytes=LOCAL_DISK_RESERVE_BYTES,
            label="full_prepare (metadata-only)",
        ), indent=2))

        # Downloaded shard paths are tracked so the HF *blob* (not just the
        # snapshot symlink) can be deleted after each shard; otherwise 71GB of
        # parquet accumulates and the local disk runs out mid-prepare.
        downloaded_shards = {}
        shard_reader = make_hf_parquet_stream_reader(
            cache_dir=HF_PARQUET_CACHE_DIR,
            batch_size=FULL_PARQUET_BATCH_SIZE,
            downloaded=downloaded_shards,
        )

        def drop_shard_download(ref):
            path = downloaded_shards.pop(ref.shard_key, None)
            freed = cleanup_shard_download(path) if path else 0
            if freed:
                print(f"  freed {freed / 1e9:.2f}GB after shard {ref.shard_key}")
            return freed

        prep = run_full_prepare_streaming(
            splits={"train": train_effective, "validation": val_effective},
            state_dir=FULL_STATE_DIR,
            vocab=vocab,
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            shard_reader=shard_reader,
            local_audio_cache_dir=LOCAL_AUDIO_CACHE_DIR,
            target_sr=TARGET_SAMPLING_RATE,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            resume=True,
            shard_bytes=shard_sizes,
            cleanup_shard=drop_shard_download,
            write_audio=False,
        )
        train_res, val_res = prep["train"], prep["validation"]
        shard_info = prep["_shards"]
        print(
            f"shards: {len(shard_info['completed'])}/{len(shard_info['plans'])} complete, "
            f"opened this session={len(shard_info['opened'])}, "
            f"audio_written={shard_info['audio_written']}"
        )
        if not train_res.get("accounting_ok") or not val_res.get("accounting_ok"):
            raise RuntimeError(
                f"Prepare UID accounting failed: train={train_res.get('accounting_error')} "
                f"val={val_res.get('accounting_error')}"
            )
        # Canonical prepare contract, stored in the summary so later stages bind
        # to it field by field instead of trusting a status string.
        prepare_contract = prepare_contract_early
        full_summary = finalize_full_prepare(
            state_dir=FULL_STATE_DIR, train_result=train_res, val_result=val_res,
            data_contract=prepare_contract,
            dataset_id=DATASET_ID, dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            expected_train_clean_count=len(train_effective),
            expected_validation_clean_count=len(val_effective),
        )
        full_stage_contract = prepare_contract
        full_status = full_summary["status"]
        print(json.dumps(full_summary, indent=2))
        if full_status != "SUCCESS_FULL_PREPARE":
            raise RuntimeError(f"Full prepare incomplete: {full_summary}")
    except Exception as _e:
        fail_stage("full_prepare", _e)


In [ ]:
# Cell F2 — FULL_STAGE=resume_test_a / resume_test_b (TWO RunPod sessions/pods, never one)
if can_continue and RUN_MODE == "full" and FULL_STAGE in {"resume_test", "resume_test_a", "resume_test_b"}:
    try:
        print("=" * 60); print(f"FULL RESUME TEST ({FULL_STAGE})"); print("=" * 60)
        if FULL_STAGE == "resume_test":
            raise RuntimeError(
                "Single-session resume_test is not a valid proof: Phase B must run in a new "
                "process. Run FULL_STAGE='resume_test_a', restart the runtime, then "
                "FULL_STAGE='resume_test_b'."
            )
        from src.asr_full_data import (
            cleanup_shard_download,
            hydrate_rows_audio,
            make_hf_parquet_stream_reader,
        )
        from src.asr_full_train import (
            assert_cross_session_resume,
            derive_resume_test_status_from_proof,
            make_resume_proof_callback,
            make_sequential_sampler_trainer_cls,
            make_uid_tracking_collator,
            make_uid_tracking_trainer_cls,
            plan_expected_resume_position,
            new_session_token,
            summarize_resume_proof,
        )

        train_effective, _pair_excl, _pair_rep = drop_train_pair_key_overlaps(
            train_clean_df, val_clean_df,
        )
        val_effective = val_clean_df
        train_mfp = compute_manifest_content_hash(train_effective)
        val_mfp = compute_manifest_content_hash(val_effective)
        # One canonical contract, built before the gate and reused for it: prepare
        # must match every field, not just a status and a couple of hashes.
        data_contract = build_data_contract(
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=train_mfp,
            validation_manifest_content_hash=val_mfp,
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
        )
        FULL_STATE_DIR = RUNTIME_PATHS.prepare_state_dir(data_contract["contract_hash"])
        print(f"FULL_STATE_DIR (contract-scoped)={FULL_STATE_DIR}")
        if not load_prepare_success(FULL_STATE_DIR, expected_contract=data_contract):
            raise RuntimeError(
                "resume_test requires SUCCESS_FULL_PREPARE bound to the current "
                "canonical data contract (dataset/parquet/manifests/vocab/audio/model)"
            )
        assert_no_frozen_test_access(list(OPENED_PATHS), list(LOADED_SPLITS))
        train_elig_path = register_path(FULL_STATE_DIR / "full_train_eligible.csv")
        train_elig = pd.read_csv(train_elig_path)
        assert_eligible_frames_no_frozen_splits(train_elig, label="full_train_eligible")
        subset = build_resume_test_subset(
            train_elig, n_samples=RESUME_TEST_SUBSET_SIZE, seed=SEED,
        )

        # A fresh session has no local WAVs: rematerialize just this subset's shards.
        LOCAL_AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        HF_PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        downloaded_shards = {}
        shard_reader = make_hf_parquet_stream_reader(
            cache_dir=HF_PARQUET_CACHE_DIR,
            batch_size=FULL_PARQUET_BATCH_SIZE,
            downloaded=downloaded_shards,
        )

        def drop_shard_download(ref):
            path = downloaded_shards.pop(ref.shard_key, None)
            return cleanup_shard_download(path) if path else 0

        hydrate_report = hydrate_rows_audio(
            subset,
            state_dir=FULL_STATE_DIR,
            dataset_id=DATASET_ID,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            shard_reader=shard_reader,
            local_audio_dir=LOCAL_AUDIO_CACHE_DIR,
            target_sr=TARGET_SAMPLING_RATE,
            cleanup_shard=drop_shard_download,
        )
        print(f"hydrate: {hydrate_report['hydrated']} files across {hydrate_report['shards']} shards")
        cache_roots = [LOCAL_AUDIO_CACHE_DIR, AUDIO_CACHE_DIR]
        assert_eligible_audio_available(
            subset, cache_roots,
            dataset_revision=EXPECTED_DATASET_REVISION,
            target_sr=TARGET_SAMPLING_RATE,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
        )

        resume_test_contract = build_resume_test_contract(
            experiment_id=FULL_EXPERIMENT_ID, data_contract=data_contract,
            subset_size=RESUME_TEST_SUBSET_SIZE, seed=SEED,
        )
        training_contract = resume_test_contract
        full_stage_contract = resume_test_contract
        session_token = new_session_token()

        from torch.utils.data import Dataset
        import soundfile as sf
        from transformers import Trainer, TrainingArguments, Wav2Vec2ForCTC
        from src.asr_full_train import list_step_checkpoints

        class FullEligibleASRDataset(Dataset):
            def __init__(self, df, vocab, cache_roots, processor, target_sr=TARGET_SAMPLING_RATE):
                self.df = df.reset_index(drop=True)
                self.vocab = vocab
                self.processor = processor
                self.target_sr = target_sr
                self.paths = []
                for _, row in self.df.iterrows():
                    p = resolve_eligible_audio_path(row, cache_roots)
                    if p is None:
                        raise RuntimeError(f"missing audio for {row['record_uid']}")
                    self.paths.append(p)
                self.df["text_norm"] = self.df["text_bahnar"].apply(normalize_bahnar_ctc_v1)

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                wav, sr = sf.read(self.paths[idx], dtype="float32")
                input_values = preprocess_features_for_training(
                    self.processor, wav, sr, self.target_sr
                )
                labels = encode_text_with_vocab(row["text_bahnar"], self.vocab)
                return {
                    "input_values": input_values,
                    "labels": labels,
                    "record_uid": str(row["record_uid"]),
                }

        ds = FullEligibleASRDataset(subset, vocab, cache_roots, processor)
        data_collator = CTCDataCollatorWithPadding(processor=processor)
        # Both phases iterate the subset in the same fixed order, otherwise "the
        # first sample after resume" has no defined value to check against.
        SequentialSamplerTrainer = make_sequential_sampler_trainer_cls(Trainer)

        rt_dir = experiment_checkpoint_dir(
            LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
        )
        restore_experiment_checkpoints_from_durable(
            rt_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
            expected_contract=resume_test_contract,
        )
        rt_dir.mkdir(parents=True, exist_ok=True)
        ensure_experiment_fingerprint(
            rt_dir, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
            allow_create_if_empty=True,
        )
        if not list(rt_dir.glob("checkpoint-*")):
            write_checkpoint_fingerprint(
                rt_dir, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                global_step=0, overwrite=False, extra=resume_test_contract,
            )

        phase_a_path = FULL_STATE_DIR / "full_resume_test_phase_a.json"
        DURABLE_CHECKPOINT_BUDGET_BYTES = resolve_durable_checkpoint_budget_bytes()

        if FULL_STAGE == "resume_test_a":
            model_a = Wav2Vec2ForCTC.from_pretrained(
                PRETRAINED_MODEL_ID, revision=PRETRAINED_MODEL_REVISION,
                ctc_loss_reduction="mean", pad_token_id=tokenizer.pad_token_id,
                vocab_size=len(tokenizer), ignore_mismatched_sizes=True,
            )
            model_a.freeze_feature_encoder()
            model_a.gradient_checkpointing_enable()
            model_a.to(device)
            args_a = TrainingArguments(
                output_dir=str(rt_dir), max_steps=RESUME_TEST_PHASE_A_STEPS,
                per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=LEARNING_RATE, fp16=bool(IS_CUDA),
                gradient_checkpointing=True, save_steps=RESUME_TEST_PHASE_A_STEPS,
                save_total_limit=2, save_only_model=False, report_to=[],
                logging_steps=10, seed=SEED, remove_unused_columns=False,
                # Deterministic order is what makes the resume position provable.
                group_by_length=False, dataloader_num_workers=0,
                dataloader_drop_last=False, ignore_data_skip=False,
            )
            trainer_a = SequentialSamplerTrainer(
                model=model_a, args=args_a, train_dataset=ds, data_collator=data_collator,
            )
            out_a = trainer_a.train()
            step_a = int(getattr(out_a, "global_step", 0) or trainer_a.state.global_step)
            ckpt_a = rt_dir / f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            if not ckpt_a.exists():
                cands = list_step_checkpoints(rt_dir)
                ckpt_a = cands[-1] if cands else None
            if ckpt_a is None:
                raise RuntimeError("Phase A did not produce a checkpoint")
            info_a = assert_checkpoint_complete_for_resume(ckpt_a)
            write_checkpoint_fingerprint(
                rt_dir, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                global_step=step_a, overwrite=True, extra=resume_test_contract,
                preserve_existing=True,
            )
            sync_experiment_checkpoints_to_durable(
                rt_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
            )
            phase_a_payload = {
                "status": "PHASE_A_COMPLETE",
                "experiment_id": FULL_EXPERIMENT_ID,
                "checkpoint": str(ckpt_a),
                "checkpoint_name": Path(ckpt_a).name,
                "global_step": step_a,
                "reached_target": step_a >= RESUME_TEST_PHASE_A_STEPS,
                "session": session_token,
                "resume_test_contract": resume_test_contract,
                "data_contract": data_contract,
                # The exact sample Phase B must consume first, derived from the
                # sequential order Phase A actually used.
                "resume_position": plan_expected_resume_position(
                    subset["record_uid"].astype(str).tolist(),
                    resume_step=step_a,
                    per_device_train_batch_size=1,
                    gradient_accumulation_steps=8,
                    dataloader_length=len(subset),  # batch_size=1, drop_last=False
                ),
                "artifacts": {
                    k: info_a[k]
                    for k in ["global_step", "model_ok", "optimizer_ok", "scheduler_ok", "rng_ok"]
                },
            }
            phase_a_path.write_text(
                json.dumps(phase_a_payload, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            del trainer_a, model_a
            full_status = "SUCCESS_FULL_RESUME_TEST_PHASE_A"
            print(json.dumps(phase_a_payload, indent=2))
            print("Phase A durable — RESTART the runtime, then set FULL_STAGE='resume_test_b'")
        else:
            if not phase_a_path.is_file():
                raise RuntimeError(
                    "resume_test_b requires full_resume_test_phase_a.json from a prior session"
                )
            phase_a_payload = json.loads(phase_a_path.read_text(encoding="utf-8"))
            assert_training_contract(
                extract_training_contract(phase_a_payload),
                resume_test_contract,
                label="resume_test phase A artifact",
            )
            cross_session = assert_cross_session_resume(
                phase_a_payload, current_session=session_token
            )
            step_a = int(phase_a_payload.get("global_step") or RESUME_TEST_PHASE_A_STEPS)
            ckpt_name = phase_a_payload.get("checkpoint_name") or f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            ckpt_a = rt_dir / ckpt_name
            if not ckpt_a.exists():
                raise RuntimeError(f"Phase B missing restored checkpoint {ckpt_a}")
            info_a = assert_checkpoint_complete_for_resume(ckpt_a)

            # New model shell in this new process; Trainer must restore everything.
            model_b = Wav2Vec2ForCTC.from_pretrained(
                PRETRAINED_MODEL_ID, revision=PRETRAINED_MODEL_REVISION,
                ctc_loss_reduction="mean", pad_token_id=tokenizer.pad_token_id,
                vocab_size=len(tokenizer), ignore_mismatched_sizes=True,
            )
            model_b.freeze_feature_encoder()
            model_b.gradient_checkpointing_enable()
            model_b.to(device)
            args_b = TrainingArguments(
                output_dir=str(rt_dir), max_steps=RESUME_TEST_PHASE_B_STEPS,
                per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=LEARNING_RATE, fp16=bool(IS_CUDA),
                gradient_checkpointing=True, save_steps=RESUME_TEST_PHASE_B_STEPS,
                save_total_limit=2, save_only_model=False, report_to=[],
                logging_steps=10, seed=SEED, remove_unused_columns=False,
                group_by_length=False, dataloader_num_workers=0,
                dataloader_drop_last=False, ignore_data_skip=False,
            )
            # Proof is captured inside the callback at on_train_begin / first
            # on_step_begin — comparing state after training continued would
            # prove nothing about the restore itself.
            proof_sink = {}
            position_sink = {}
            # UIDs are recorded at collate time and claimed by the first
            # training_step: batches that resume skips also hit the collator and
            # Dataset.__getitem__, so only training_step proves consumption.
            tracking_collator = make_uid_tracking_collator(data_collator, position_sink)
            TrackingTrainer = make_uid_tracking_trainer_cls(
                SequentialSamplerTrainer, position_sink,
            )
            trainer_b = TrackingTrainer(
                model=model_b, args=args_b, train_dataset=ds,
                data_collator=tracking_collator,
                callbacks=[make_resume_proof_callback(
                    checkpoint_dir=ckpt_a,
                    expected_resume_step=step_a,
                    sink=proof_sink,
                    gradient_accumulation_steps=8,
                )],
            )
            out_b = trainer_b.train(resume_from_checkpoint=str(ckpt_a))
            step_b = int(getattr(out_b, "global_step", 0) or trainer_b.state.global_step)
            expected_position = phase_a_payload.get("resume_position") or {}
            proof_sink.update(position_sink)
            proof = summarize_resume_proof(
                proof_sink,
                expected_first_uid=expected_position.get("expected_first_uid"),
            )
            proof["final_global_step"] = step_b
            ckpt_b = rt_dir / f"checkpoint-{RESUME_TEST_PHASE_B_STEPS}"
            if not ckpt_b.exists():
                cands = list_step_checkpoints(rt_dir)
                ckpt_b = cands[-1] if cands else None
            if ckpt_b is None:
                raise RuntimeError("Phase B did not produce a checkpoint")
            info_b = assert_checkpoint_complete_for_resume(ckpt_b)
            write_checkpoint_fingerprint(
                rt_dir, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                global_step=step_b, overwrite=True, extra=resume_test_contract,
                preserve_existing=True,
            )
            durable_rt = sync_experiment_checkpoints_to_durable(
                rt_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
            )
            verdict = derive_resume_test_status_from_proof(
                proof,
                phase_a_reached_target=bool(phase_a_payload.get("reached_target"))
                and step_a >= RESUME_TEST_PHASE_A_STEPS,
                cross_session=cross_session,
                used_separate_experiment_dir=(RESUME_TEST_MARKER in str(rt_dir)),
                expected_phase_b_steps=RESUME_TEST_PHASE_B_STEPS,
            )
            status = verdict["status"]
            payload = {
                "status": status,
                "experiment_id": FULL_EXPERIMENT_ID,
                "dataset_id": DATASET_ID,
                "dataset_revision": EXPECTED_DATASET_REVISION,
                "parquet_revision": EXPECTED_PARQUET_REVISION,
                "vocab_fp": vocab_fingerprint(vocab),
                "pretrained_model_id": PRETRAINED_MODEL_ID,
                "pretrained_model_revision": PRETRAINED_MODEL_REVISION,
                "phase_a_step": step_a,
                "phase_b_step": step_b,
                "checkpoint_dir": str(durable_rt or rt_dir),
                "phase_a_artifacts": phase_a_payload.get("artifacts") or info_a,
                "phase_b_artifacts": {
                    k: info_b[k]
                    for k in ["global_step", "model_ok", "optimizer_ok", "scheduler_ok", "rng_ok"]
                },
                "subset_size": len(subset),
                "true_restart": bool(verdict["checks"]["true_restart"]),
                "cross_session": cross_session,
                "checks": verdict["checks"],
                "failed_checks": verdict["failed_checks"],
                "proof": proof,
                "resume_test_contract": resume_test_contract,
                "data_contract": data_contract,
                "train_manifest_content_hash": data_contract["train_manifest_content_hash"],
                "validation_manifest_content_hash": data_contract["validation_manifest_content_hash"],
                "processing_version": EXPECTED_AUDIO_PROCESSING_VERSION,
                "min_duration": MIN_AUDIO_DURATION,
                "max_duration": MAX_AUDIO_DURATION,
                "target_sr": TARGET_SAMPLING_RATE,
                "note": "resume_test checkpoints must NEVER initialize full training",
            }
            write_resume_test_summary(FULL_STATE_DIR, payload)
            full_status = status
            print(json.dumps(payload, indent=2))
            if status != "SUCCESS_FULL_RESUME_TEST":
                raise RuntimeError(f"Resume test failed on {verdict['failed_checks']}: {payload}")
    except SystemExit:
        raise
    except Exception as _e:
        fail_stage("full_resume_test", _e)


### Optional GPU memory probe (MAX=40)

Manual only — set `RUN_GPU_PROBE = True` and run this cell alone. Not part of Run-all / `FULL_STAGE`.


In [ ]:
# Optional — NOT executed by FULL_STAGE gates
RUN_GPU_PROBE = False  # flip to True manually on a GPU runtime
if RUN_GPU_PROBE:
    from src.asr_full_train import probe_gpu_memory_for_duration
    from transformers import Wav2Vec2ForCTC
    _probe_model = Wav2Vec2ForCTC.from_pretrained(
        PRETRAINED_MODEL_ID, revision=PRETRAINED_MODEL_REVISION,
        ctc_loss_reduction="mean", pad_token_id=tokenizer.pad_token_id,
        vocab_size=len(tokenizer), ignore_mismatched_sizes=True,
    )
    _probe_model.gradient_checkpointing_enable()
    print(json.dumps(probe_gpu_memory_for_duration(
        model=_probe_model, processor=processor,
        duration_seconds=MAX_AUDIO_DURATION, target_sr=TARGET_SAMPLING_RATE,
        device=device, vocab_size=len(tokenizer),
    ), indent=2))
    del _probe_model
else:
    print("GPU probe skipped (RUN_GPU_PROBE=False)")


In [ ]:
# Cell F3 — FULL_STAGE=train (from locked base model; resume only same FULL_EXPERIMENT_ID)
if can_continue and RUN_MODE == "full" and FULL_STAGE == "train":
    try:
        print("=" * 60); print("FULL TRAIN"); print("=" * 60)
        train_effective, _pair_excl, _pair_rep = drop_train_pair_key_overlaps(
            train_clean_df, val_clean_df,
        )
        val_effective = val_clean_df
        train_mfp = compute_manifest_content_hash(train_effective)
        val_mfp = compute_manifest_content_hash(val_effective)
        # Canonical contract + contract-scoped state BEFORE any summary/CSV/gate.
        data_contract = build_data_contract(
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=train_mfp,
            validation_manifest_content_hash=val_mfp,
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
        )
        FULL_STATE_DIR = RUNTIME_PATHS.prepare_state_dir(data_contract["contract_hash"])
        print(f"FULL_STATE_DIR (contract-scoped)={FULL_STATE_DIR}")
        training_contract = build_training_contract(
            experiment_id=FULL_EXPERIMENT_ID,
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=train_mfp,
            validation_manifest_content_hash=val_mfp,
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
            hparams={},  # filled after hp build
        )
        full_stage_contract = training_contract
        assert_ready_for_full_train(
            FULL_STATE_DIR,
            expected_contract=training_contract,
        )
        assert_no_frozen_test_access(list(OPENED_PATHS), list(LOADED_SPLITS))

        train_elig_path = register_path(FULL_STATE_DIR / "full_train_eligible.csv")
        val_elig_path = register_path(FULL_STATE_DIR / "full_validation_eligible.csv")
        train_elig = pd.read_csv(train_elig_path)
        val_elig = pd.read_csv(val_elig_path)
        cache_roots = [LOCAL_AUDIO_CACHE_DIR, AUDIO_CACHE_DIR]
        assert_eligible_frames_no_frozen_splits(train_elig, label="full_train_eligible")
        assert_eligible_frames_no_frozen_splits(val_elig, label="full_validation_eligible")

        # Local audio is disposable: a restarted RunPod session rehydrates it from
        # the pinned parquet snapshot, verifying each file against its stored
        # PCM hash. Completed shards are never re-QA'd.
        from src.asr_full_data import (
            assert_local_disk_budget,
            estimate_hydrate_wav_bytes,
            cleanup_shard_download,
            expected_wav_bytes,
            hydrate_union_audio,
            make_hf_parquet_stream_reader,
        )
        LOCAL_AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        HF_PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        downloaded_shards = {}
        shard_reader = make_hf_parquet_stream_reader(
            cache_dir=HF_PARQUET_CACHE_DIR,
            batch_size=FULL_PARQUET_BATCH_SIZE,
            downloaded=downloaded_shards,
        )

        def drop_shard_download(ref):
            path = downloaded_shards.pop(ref.shard_key, None)
            return cleanup_shard_download(path) if path else 0

        # Budget measured from the artifacts themselves, never from a hardcoded
        # hours/GB figure, and checked before a single byte is downloaded.
        _wav = expected_wav_bytes([train_elig, val_elig])
        _ckpt_estimate = estimate_checkpoint_bytes(
            n_parameters=FULL_MODEL_PARAM_COUNT, save_only_model=False,
        )
        print(
            f"audio to hydrate: {_wav['unique_records']} records, "
            f"{_wav['audio_hours']:.1f}h, {_wav['wav_bytes'] / 1e9:.1f}GB"
        )
        # Single WAV estimate (union); do not double-count hydrate + hydrated.
        wav_need = int(_wav["wav_bytes"])
        _existing_ckpt_bytes = measure_dir_bytes(experiment_checkpoint_dir(
            LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
        ))
        _peak = plan_local_checkpoint_disk_peak(
            existing_checkpoint_bytes=_existing_ckpt_bytes,
            new_checkpoint_bytes=_ckpt_estimate["checkpoint_bytes"],
            hydrated_wav_bytes=wav_need,
        )
        print(json.dumps(assert_local_disk_budget(
            LOCAL_AUDIO_CACHE_DIR,
            needs={
                "hydrated_wav": wav_need,
                "largest_parquet_shard": FULL_SHARD_BYTES_HINT,
                "hf_model_cache": LOCAL_HF_MODEL_CACHE_BYTES,
                "local_checkpoint_peak": _peak["checkpoint_peak_bytes"],
            },
            reserve_bytes=LOCAL_DISK_RESERVE_BYTES,
            label="full_train hydrate+train",
        ), indent=2))

        # Train and validation come from the same physical shards: hydrating the
        # union downloads each shard once instead of twice.
        _rep = hydrate_union_audio(
            [train_elig, val_elig],
            state_dir=FULL_STATE_DIR,
            dataset_id=DATASET_ID,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            shard_reader=shard_reader,
            local_audio_dir=LOCAL_AUDIO_CACHE_DIR,
            target_sr=TARGET_SAMPLING_RATE,
            cleanup_shard=drop_shard_download,
        )
        print(
            f"hydrate[union]: {_rep['hydrated']} files for "
            f"{_rep['unique_records']} records over {_rep['shards']} shards"
        )
        assert_eligible_audio_available(
            train_elig, cache_roots,
            dataset_revision=EXPECTED_DATASET_REVISION,
            target_sr=TARGET_SAMPLING_RATE,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
        )
        assert_eligible_audio_available(
            val_elig, cache_roots,
            dataset_revision=EXPECTED_DATASET_REVISION,
            target_sr=TARGET_SAMPLING_RATE,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
        )

        hp = build_full_training_hparams(
            n_train=len(train_elig), num_train_epochs=FULL_NUM_TRAIN_EPOCHS,
            save_steps=FULL_SAVE_STEPS, save_total_limit=FULL_SAVE_TOTAL_LIMIT,
            fp16=bool(IS_CUDA), gradient_checkpointing=True, seed=SEED,
        )
        training_contract["hparams"] = hp
        train_contract = build_train_contract(
            experiment_id=FULL_EXPERIMENT_ID, data_contract=data_contract, hparams=hp,
        )
        training_contract = train_contract
        val_monitor_df = build_validation_monitor_subset(
            val_elig, n_samples=FULL_VAL_MONITOR_SIZE, seed=SEED,
        )

        from torch.utils.data import Dataset
        import soundfile as sf
        from transformers import Trainer, TrainingArguments, Wav2Vec2ForCTC

        class FullEligibleASRDataset(Dataset):
            def __init__(self, df, vocab, cache_roots, processor, target_sr=TARGET_SAMPLING_RATE):
                self.df = df.reset_index(drop=True)
                self.vocab = vocab
                self.processor = processor
                self.target_sr = target_sr
                self.paths = []
                for _, row in self.df.iterrows():
                    p = resolve_eligible_audio_path(row, cache_roots)
                    if p is None:
                        raise RuntimeError(f"missing audio for {row['record_uid']}")
                    self.paths.append(p)
                self.df["text_norm"] = self.df["text_bahnar"].apply(normalize_bahnar_ctc_v1)

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                wav, sr = sf.read(self.paths[idx], dtype="float32")
                input_values = preprocess_features_for_training(
                    self.processor, wav, sr, self.target_sr
                )
                labels = encode_text_with_vocab(row["text_bahnar"], self.vocab)
                return {
                    "input_values": input_values,
                    "labels": labels,
                    "record_uid": str(row["record_uid"]),
                    "reference_raw": row["text_bahnar"],
                    "reference_normalized": row["text_norm"],
                }

        train_dataset = FullEligibleASRDataset(train_elig, vocab, cache_roots, processor)
        val_monitor_dataset = FullEligibleASRDataset(val_monitor_df, vocab, cache_roots, processor)
        val_full_dataset = FullEligibleASRDataset(val_elig, vocab, cache_roots, processor)
        data_collator = CTCDataCollatorWithPadding(processor=processor)

        exp_dir = experiment_checkpoint_dir(
            LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
        )
        restore_experiment_checkpoints_from_durable(
            exp_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            expected_contract=training_contract,
        )
        exp_dir.mkdir(parents=True, exist_ok=True)
        # Fail-closed: existing checkpoints must already carry a fingerprint that
        # matches this exact train contract; nothing is stamped retroactively.
        ensure_experiment_fingerprint(
            exp_dir, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            expected_contract=training_contract, allow_create_if_empty=True,
        )
        if not list((exp_dir).glob('checkpoint-*')):
            write_checkpoint_fingerprint(
                exp_dir, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
                global_step=0, overwrite=False, extra=training_contract,
            )
        resume_ck = resolve_resume_checkpoint(
            resume_policy=RESUME_POLICY, experiment_dir=exp_dir,
            experiment_id=FULL_EXPERIMENT_ID, explicit_path=FULL_RESUME_CHECKPOINT,
            kind=FULL_TRAIN_MARKER,
        )
        if resume_ck:
            assert_checkpoint_allowed_for_full_train(
                resume_ck, experiment_id=FULL_EXPERIMENT_ID, expected_contract=training_contract,
                experiment_root=exp_dir,
            )
            started_from_base = True
        else:
            started_from_base = (PRETRAINED_MODEL_ID == "facebook/wav2vec2-xls-r-300m")
            if not started_from_base:
                raise RuntimeError("Full training must start from facebook/wav2vec2-xls-r-300m")
        # Always construct model shell; Trainer loads weights from resume_ck when set
        model = Wav2Vec2ForCTC.from_pretrained(
            PRETRAINED_MODEL_ID, revision=PRETRAINED_MODEL_REVISION,
            ctc_loss_reduction="mean", pad_token_id=tokenizer.pad_token_id,
            vocab_size=len(tokenizer), ignore_mismatched_sizes=True,
        )
        model.freeze_feature_encoder()
        model.gradient_checkpointing_enable()
        model.to(device)

        def compute_metrics(eval_pred):
            logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
            labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            pred_ids = np.argmax(logits, axis=-1)
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            refs = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
            hyps = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
            m = compute_batch_metrics(refs, hyps, allow_empty_reference=False)
            return {"cer": m["cer"], "wer": m["wer"]}

        args = TrainingArguments(
            output_dir=str(exp_dir),
            max_steps=hp["max_steps"],
            per_device_train_batch_size=hp["per_device_train_batch_size"],
            per_device_eval_batch_size=hp["per_device_eval_batch_size"],
            gradient_accumulation_steps=hp["gradient_accumulation_steps"],
            learning_rate=hp["learning_rate"],
            warmup_ratio=hp["warmup_ratio"],
            fp16=hp["fp16"],
            gradient_checkpointing=hp["gradient_checkpointing"],
            save_steps=hp["save_steps"],
            save_total_limit=hp["save_total_limit"],
            save_only_model=False,
            eval_strategy="steps",
            eval_steps=hp["save_steps"],
            load_best_model_at_end=True,
            metric_for_best_model="cer",
            greater_is_better=False,
            report_to=[],
            seed=hp["seed"],
            logging_steps=50,
            remove_unused_columns=False,
            dataloader_pin_memory=bool(IS_CUDA),
        )
        print(json.dumps({k: hp[k] for k in ["steps_per_epoch", "max_steps", "num_train_epochs"]}, indent=2))
        print(f"resume_from={resume_ck}")

        DURABLE_CHECKPOINT_BUDGET_BYTES = resolve_durable_checkpoint_budget_bytes()
        durable_sync_cb = make_durable_checkpoint_sync_callback(
            local_experiment_dir=exp_dir, full_state_dir=FULL_STATE_DIR,
            experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            training_contract=training_contract,
            save_total_limit=int(hp["save_total_limit"]),
            budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
        )
        trainer = Trainer(
            model=model, args=args, train_dataset=train_dataset,
            eval_dataset=val_monitor_dataset, data_collator=data_collator,
            compute_metrics=compute_metrics, processing_class=processor,
            callbacks=[durable_sync_cb],
        )
        # Exact parameter count is only available once the shell exists; durable storage
        # budget is validated here so a doomed run fails now rather than at the
        # first save, hours in.
        _ckpt_exact = estimate_checkpoint_bytes(
            n_parameters=int(model.num_parameters()), save_only_model=False,
        )
        print(json.dumps(assert_durable_checkpoint_budget(
            protected_bytes=durable_experiment_protected_bytes(
                FULL_STATE_DIR, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            ),
            upload_bytes=_ckpt_exact["checkpoint_bytes"],
            budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
            label=f"pre-train estimate {FULL_EXPERIMENT_ID}",
            detail={
                "n_parameters": _ckpt_exact["n_parameters"],
                "per_checkpoint_bytes": _ckpt_exact["checkpoint_bytes"],
                "estimate_scope": "one_pending_checkpoint",
            },
        ), indent=2))

        t0 = time.perf_counter()
        # Peak local disk again right before train: existing ckpts + one new temp.
        _existing_ckpt_bytes = measure_dir_bytes(exp_dir)
        _peak_train = plan_local_checkpoint_disk_peak(
            existing_checkpoint_bytes=_existing_ckpt_bytes,
            new_checkpoint_bytes=_ckpt_estimate["checkpoint_bytes"],
            hydrated_wav_bytes=0,  # WAV already hydrated; do not recount
        )
        print(json.dumps(assert_local_disk_budget(
            LOCAL_AUDIO_CACHE_DIR,
            needs={"local_checkpoint_peak": _peak_train["checkpoint_peak_bytes"]},
            reserve_bytes=LOCAL_DISK_RESERVE_BYTES,
            label="full_train before trainer.train",
        ), indent=2))
        train_out = trainer.train(resume_from_checkpoint=resume_ck)
        train_runtime_seconds = float(time.perf_counter() - t0)
        global_step = int(getattr(train_out, "global_step", 0) or trainer.state.global_step)
        write_checkpoint_fingerprint(
            exp_dir, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            global_step=global_step, overwrite=True, extra=training_contract, preserve_existing=True,
        )
        sync_experiment_checkpoints_to_durable(
            exp_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            save_total_limit=FULL_SAVE_TOTAL_LIMIT,
            budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
        )

        # End-of-epoch / final full eligible validation
        full_pred = trainer.predict(val_full_dataset)
        full_metrics = dict(full_pred.metrics or {})
        full_cer = float(full_metrics.get("test_cer", full_metrics.get("eval_cer", float("nan"))))
        full_wer = float(full_metrics.get("test_wer", full_metrics.get("eval_wer", float("nan"))))
        full_validation_ok = bool(np.isfinite(full_cer) and np.isfinite(full_wer))

        history_rows = []
        for log in list(getattr(trainer.state, "log_history", []) or []):
            if "eval_cer" in log or "cer" in log:
                history_rows.append({
                    "step": log.get("step") or log.get("global_step"),
                    "cer": log.get("eval_cer", log.get("cer")),
                    "wer": log.get("eval_wer", log.get("wer")),
                    "loss": log.get("eval_loss", log.get("loss")),
                    "checkpoint": str(trainer.state.best_model_checkpoint) if trainer.state.best_model_checkpoint else None,
                })
        history_rows.append({
            "step": global_step, "cer": full_cer, "wer": full_wer,
            "loss": full_metrics.get("test_loss", full_metrics.get("eval_loss")),
            "checkpoint": str(trainer.state.best_model_checkpoint) if trainer.state.best_model_checkpoint else None,
            "scope": "full_validation",
        })
        best_row = select_best_checkpoint_by_cer(history_rows)
        best_ckpt = trainer.state.best_model_checkpoint or (
            best_row.get("checkpoint") if best_row else None
        )
        if not best_ckpt:
            from src.asr_full_train import find_latest_valid_checkpoint
            latest = find_latest_valid_checkpoint(exp_dir, experiment_id=FULL_EXPERIMENT_ID)
            best_ckpt = str(latest) if latest else None
        if not best_ckpt:
            raise RuntimeError("No best checkpoint after full training")
        assert_checkpoint_allowed_for_full_train(
            best_ckpt, experiment_id=FULL_EXPERIMENT_ID,
            expected_contract=training_contract, experiment_root=exp_dir,
        )

        # Reload best checkpoint to confirm integrity
        reloaded = Wav2Vec2ForCTC.from_pretrained(best_ckpt)
        best_checkpoint_reload_ok = reloaded is not None
        del reloaded

        # The best checkpoint is pinned into a final snapshot so retention can
        # never garbage-collect the artifact evaluate depends on. Replacing the
        # whole durable experiment tree here would destroy the snapshot store.
        sync_experiment_checkpoints_to_durable(
            exp_dir, FULL_STATE_DIR, experiment_id=FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            best_checkpoint_name=Path(best_ckpt).name,
            save_total_limit=FULL_SAVE_TOTAL_LIMIT,
            budget_bytes=DURABLE_CHECKPOINT_BUDGET_BYTES,
        )

        reached = global_step >= int(hp["max_steps"])
        frozen_hits_full = detect_frozen_leakage(
            opened_paths=OPENED_PATHS, loaded_splits=LOADED_SPLITS,
            source_splits=(list(train_elig.get("source_split", pd.Series(dtype=str)).astype(str))
                          + list(val_elig.get("source_split", pd.Series(dtype=str)).astype(str))),
            split_values=(list(train_elig.get("split", pd.Series(dtype=str)).astype(str))
                         + list(val_elig.get("split", pd.Series(dtype=str)).astype(str))),
        )
        status = derive_full_training_status(
            reached_target_steps=reached,
            full_validation_ok=full_validation_ok,
            best_checkpoint_reload_ok=best_checkpoint_reload_ok,
            frozen_test_accessed=bool(frozen_hits_full),
            started_from_base_model=started_from_base,
        )
        payload = {
            "status": status,
            "experiment_id": FULL_EXPERIMENT_ID,
            "dataset_id": DATASET_ID,
            "dataset_revision": EXPECTED_DATASET_REVISION,
            "vocab_fp": vocab_fingerprint(vocab),
            "pretrained_model_id": PRETRAINED_MODEL_ID,
            "pretrained_model_revision": PRETRAINED_MODEL_REVISION,
            "hparams": hp,
            "global_step": global_step,
            "target_steps": hp["max_steps"],
            "resume_from": resume_ck,
            "best_checkpoint": str(best_ckpt),
            "best_row": best_row,
            "full_validation": {"cer": full_cer, "wer": full_wer, "metrics": full_metrics},
            "train_runtime_seconds": train_runtime_seconds,
            "gpu_memory": collect_gpu_memory_snapshot(),
            "n_train": len(train_elig),
            "n_validation": len(val_elig),
            "n_validation_monitor": len(val_monitor_df),
            "local_checkpoint_dir": str(exp_dir),
            "durable_checkpoint_dir": str(durable_experiment_dir(
                FULL_STATE_DIR, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            )),
            "history": history_rows,
            "training_contract": training_contract,
            "train_manifest_content_hash": training_contract["train_manifest_content_hash"],
            "validation_manifest_content_hash": training_contract["validation_manifest_content_hash"],
            "processing_version": EXPECTED_AUDIO_PROCESSING_VERSION,
            "min_duration": MIN_AUDIO_DURATION,
            "max_duration": MAX_AUDIO_DURATION,
            "target_sr": TARGET_SAMPLING_RATE,
            "frozen_test_accessed": bool(frozen_hits_full),
        }
        write_full_train_summary(FULL_STATE_DIR, payload)
        (FULL_STATE_DIR / "full_train_history.json").write_text(
            json.dumps(history_rows, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        full_status = status
        training_success = (status == "SUCCESS_FULL_TRAINING")
        best_checkpoint_path = Path(best_ckpt)
        best_cer = full_cer
        eval_metrics = {"cer": full_cer, "wer": full_wer}
        print(json.dumps({k: payload[k] for k in ["status", "global_step", "best_checkpoint", "full_validation"]}, indent=2))
        if status != "SUCCESS_FULL_TRAINING":
            raise RuntimeError(f"Full training incomplete: {payload['status']}")
    except Exception as _e:
        fail_stage("full_train", _e)


In [ ]:
# Cell F4 — FULL_STAGE=evaluate
if can_continue and RUN_MODE == "full" and FULL_STAGE == "evaluate":
    try:
        print("=" * 60); print("FULL EVALUATE"); print("=" * 60)
        train_effective, _pair_excl, _pair_rep = drop_train_pair_key_overlaps(
            train_clean_df, val_clean_df,
        )
        val_effective = val_clean_df
        train_mfp = compute_manifest_content_hash(train_effective)
        val_mfp = compute_manifest_content_hash(val_effective)
        # Canonical contract + contract-scoped state BEFORE any summary/CSV/gate.
        data_contract = build_data_contract(
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=train_mfp,
            validation_manifest_content_hash=val_mfp,
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
        )
        FULL_STATE_DIR = RUNTIME_PATHS.prepare_state_dir(data_contract["contract_hash"])
        print(f"FULL_STATE_DIR (contract-scoped)={FULL_STATE_DIR}")
        training_contract = build_training_contract(
            experiment_id=FULL_EXPERIMENT_ID,
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            train_manifest_content_hash=train_mfp,
            validation_manifest_content_hash=val_mfp,
            vocab_fp=vocab_fingerprint(vocab),
            processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
            audio_pcm_pipeline_version=EXPECTED_AUDIO_PCM_PIPELINE_VERSION,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            target_sr=TARGET_SAMPLING_RATE,
            pretrained_model_id=PRETRAINED_MODEL_ID,
            pretrained_model_revision=PRETRAINED_MODEL_REVISION,
            overlap_policy=OVERLAP_POLICY_PAIR_KEY_TRAIN_DROP,
        )
        assert_no_frozen_test_access(list(OPENED_PATHS), list(LOADED_SPLITS))

        train_summary_path = FULL_STATE_DIR / "full_train_summary.json"
        if not train_summary_path.is_file():
            raise RuntimeError("evaluate requires full_train_summary.json from FULL_STAGE=train")

        val_elig_path = register_path(FULL_STATE_DIR / "full_validation_eligible.csv")
        val_elig = pd.read_csv(val_elig_path)
        cache_roots = [LOCAL_AUDIO_CACHE_DIR, AUDIO_CACHE_DIR]
        assert_eligible_frames_no_frozen_splits(val_elig, label="full_validation_eligible")

        from src.asr_full_data import (
            cleanup_shard_download,
            hydrate_rows_audio,
            make_hf_parquet_stream_reader,
        )
        LOCAL_AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        HF_PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        downloaded_shards = {}
        shard_reader = make_hf_parquet_stream_reader(
            cache_dir=HF_PARQUET_CACHE_DIR,
            batch_size=FULL_PARQUET_BATCH_SIZE,
            downloaded=downloaded_shards,
        )

        def drop_shard_download(ref):
            path = downloaded_shards.pop(ref.shard_key, None)
            return cleanup_shard_download(path) if path else 0

        _rep = hydrate_rows_audio(
            val_elig,
            state_dir=FULL_STATE_DIR,
            dataset_id=DATASET_ID,
            parquet_revision=EXPECTED_PARQUET_REVISION,
            shard_reader=shard_reader,
            local_audio_dir=LOCAL_AUDIO_CACHE_DIR,
            target_sr=TARGET_SAMPLING_RATE,
            cleanup_shard=drop_shard_download,
        )
        print(f"hydrate[validation]: {_rep['hydrated']} files over {_rep['shards']} shards")
        assert_eligible_audio_available(
            val_elig, cache_roots,
            dataset_revision=EXPECTED_DATASET_REVISION,
            target_sr=TARGET_SAMPLING_RATE,
            min_duration=MIN_AUDIO_DURATION,
            max_duration=MAX_AUDIO_DURATION,
            expected_processing_version=EXPECTED_AUDIO_PROCESSING_VERSION,
        )

        train_sum = json.loads(train_summary_path.read_text(encoding="utf-8"))
        # Recompute the hparams from config instead of trusting the summary, then
        # require an exact match: validating a summary against hparams read out
        # of that same summary would always pass.
        hp_expected = build_full_training_hparams(
            n_train=len(pd.read_csv(register_path(FULL_STATE_DIR / "full_train_eligible.csv"))),
            num_train_epochs=FULL_NUM_TRAIN_EPOCHS,
            save_steps=FULL_SAVE_STEPS, save_total_limit=FULL_SAVE_TOTAL_LIMIT,
            fp16=bool(IS_CUDA), gradient_checkpointing=True, seed=SEED,
        )
        hp_recorded = train_sum.get("hparams") or (train_sum.get("train_contract") or {}).get("hparams") or {}
        if dict(hp_recorded) != dict(hp_expected):
            raise RuntimeError(
                "Full-train hparams in full_train_summary.json do not match the current "
                f"configuration: recorded={hp_recorded} expected={hp_expected}"
            )
        hp = hp_expected
        train_contract = build_train_contract(
            experiment_id=FULL_EXPERIMENT_ID, data_contract=data_contract, hparams=hp,
        )
        evaluate_contract = build_evaluate_contract(train_contract=train_contract)
        training_contract = evaluate_contract
        exp_dir = experiment_checkpoint_dir(
            LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
        )
        best = resolve_best_checkpoint_from_durable(
            FULL_STATE_DIR,
            experiment_id=FULL_EXPERIMENT_ID,
            train_summary=train_sum,
            expected_contract=train_contract,
            local_experiment_dir=exp_dir,
        )
        assert_ready_for_full_evaluate(
            FULL_STATE_DIR,
            expected_experiment_id=FULL_EXPERIMENT_ID,
            expected_contract=train_contract,
        )

        from torch.utils.data import Dataset
        import soundfile as sf
        from transformers import Trainer, TrainingArguments, Wav2Vec2ForCTC

        class FullEligibleASRDataset(Dataset):
            def __init__(self, df, vocab, cache_roots, processor, target_sr=TARGET_SAMPLING_RATE):
                self.df = df.reset_index(drop=True)
                self.vocab = vocab
                self.processor = processor
                self.target_sr = target_sr
                self.paths = [resolve_eligible_audio_path(row, cache_roots) for _, row in self.df.iterrows()]
                if any(p is None for p in self.paths):
                    raise RuntimeError("missing audio during evaluate dataset build")
                self.df["text_norm"] = self.df["text_bahnar"].apply(normalize_bahnar_ctc_v1)

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                wav, sr = sf.read(self.paths[idx], dtype="float32")
                input_values = preprocess_features_for_training(
                    self.processor, wav, sr, self.target_sr
                )
                labels = encode_text_with_vocab(row["text_bahnar"], self.vocab)
                return {"input_values": input_values, "labels": labels, "record_uid": str(row["record_uid"])}

        eval_ds = FullEligibleASRDataset(val_elig, vocab, cache_roots, processor)
        data_collator = CTCDataCollatorWithPadding(processor=processor)
        eval_model = Wav2Vec2ForCTC.from_pretrained(best)
        eval_model.to(device)

        def compute_metrics(eval_pred):
            logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
            labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            pred_ids = np.argmax(logits, axis=-1)
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            refs = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
            hyps = [normalize_bahnar_ctc_v1(x) for x in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
            m = compute_batch_metrics(refs, hyps, allow_empty_reference=False)
            return {"cer": m["cer"], "wer": m["wer"]}

        args = TrainingArguments(
            output_dir=str(PATHS["artifacts"] / "full_eval"),
            per_device_eval_batch_size=1, report_to=[], remove_unused_columns=False,
        )
        trainer = Trainer(
            model=eval_model, args=args, data_collator=data_collator,
            compute_metrics=compute_metrics, processing_class=processor,
        )
        pred = trainer.predict(eval_ds)
        metrics = dict(pred.metrics or {})
        # predict() prefixes with "test_"; fold to canonical names so every required
        # metric is checked, loss included.
        canonical_metrics = canonicalize_trainer_metrics(metrics)
        cer, wer = canonical_metrics["cer"], canonical_metrics["wer"]
        frozen_hit_eval = bool(detect_frozen_leakage(
            opened_paths=OPENED_PATHS, loaded_splits=LOADED_SPLITS,
            source_splits=list(val_elig.get("source_split", pd.Series(dtype=str)).astype(str)),
            split_values=list(val_elig.get("split", pd.Series(dtype=str)).astype(str)),
        ))
        full_stage_contract = train_contract
        verdict = derive_full_evaluate_status(
            full_train_success=load_full_train_success(
                FULL_STATE_DIR,
                expected_experiment_id=FULL_EXPERIMENT_ID,
                expected_contract=train_contract,
            ),
            best_checkpoint_from_durable_valid=Path(best).is_dir(),
            metrics_finite=metrics_are_finite(
                canonical_metrics, keys=CANONICAL_METRIC_KEYS,
            ),
            frozen_test_accessed=frozen_hit_eval,
            contract_matches=training_contract_matches(
                extract_training_contract(train_sum), train_contract, require_hparams=True
            ),
        )
        payload = {
            "status": verdict["status"],
            "checks": verdict["checks"],
            "failed_checks": verdict["failed_checks"],
            "experiment_id": FULL_EXPERIMENT_ID,
            "dataset_id": DATASET_ID,
            "dataset_revision": EXPECTED_DATASET_REVISION,
            "vocab_fp": vocab_fingerprint(vocab),
            "pretrained_model_id": PRETRAINED_MODEL_ID,
            "pretrained_model_revision": PRETRAINED_MODEL_REVISION,
            "checkpoint": str(best),
            "n_validation": len(val_elig),
            "cer": cer,
            "wer": wer,
            "metrics": metrics,
            "canonical_metrics": canonical_metrics,
            "training_contract": training_contract,
            "train_manifest_content_hash": training_contract["train_manifest_content_hash"],
            "validation_manifest_content_hash": training_contract["validation_manifest_content_hash"],
            "processing_version": EXPECTED_AUDIO_PROCESSING_VERSION,
            "min_duration": MIN_AUDIO_DURATION,
            "max_duration": MAX_AUDIO_DURATION,
            "target_sr": TARGET_SAMPLING_RATE,
            "frozen_test_accessed": frozen_hit_eval,
        }
        (FULL_STATE_DIR / "full_evaluate_summary.json").write_text(
            json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        full_status = verdict["status"]
        eval_metrics = {"cer": cer, "wer": wer}
        print(json.dumps(payload, indent=2))
        if verdict["status"] != "SUCCESS_FULL_EVALUATE":
            raise RuntimeError(f"Full evaluate failed on {verdict['failed_checks']}")
    except Exception as _e:
        fail_stage("full_evaluate", _e)


## Step 10 — Export artifacts (ALWAYS run) in a fixed order, then raise if FAILED

Order: CSVs → config/env/provenance → checkpoint metadata → validation
metrics/predictions → integrity → run_summary → training_error → manifest.

In [ ]:
# Cell 23 — (1/9) Exclusion CSVs (fixed schema, always written)
audio_excl_rows = list(empty_ref_exclusions)
audio_excl_df = (pd.DataFrame(audio_excl_rows).reindex(columns=AUDIO_EXCLUSIONS_COLUMNS)
                 if audio_excl_rows else empty_dataframe(AUDIO_EXCLUSIONS_COLUMNS))
audio_excl_df.to_csv(PATHS["artifacts"] / "audio_exclusions.csv", index=False)

ctc_df = (pd.DataFrame(ctc_exclusions).reindex(columns=CTC_FEASIBILITY_EXCLUSIONS_COLUMNS)
          if ctc_exclusions else empty_dataframe(CTC_FEASIBILITY_EXCLUSIONS_COLUMNS))
ctc_df.to_csv(PATHS["artifacts"] / "ctc_feasibility_exclusions.csv", index=False)
print(f"audio_exclusions={len(audio_excl_df)} (empty_ref) ctc_exclusions={len(ctc_df)}")


In [ ]:
# Cell 24 — (2/9) config, environment, training_history, contract checks, processor provenance, OOV, cache pool
run_config = {
    "run_id": RUN_ID, "run_mode": RUN_MODE,
    "full_training_implemented": FULL_TRAINING_IMPLEMENTED,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_id": DATASET_ID, "dataset_revision": EXPECTED_DATASET_REVISION,
    "pretrained_model_id": PRETRAINED_MODEL_ID, "pretrained_model_revision": PRETRAINED_MODEL_REVISION,
    "vocab_size": (len(tokenizer) if tokenizer is not None else None),
    "target_sampling_rate": TARGET_SAMPLING_RATE,
    "min_audio_duration": MIN_AUDIO_DURATION, "max_audio_duration": MAX_AUDIO_DURATION,
    "seed": SEED, "pilot_seed": PILOT_SEED,
    "pilot_train_samples": PILOT_TRAIN_SAMPLES, "pilot_validation_samples": PILOT_VALIDATION_SAMPLES,
    "max_steps": MAX_STEPS, "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE, "warmup_steps": WARMUP_STEPS,
    "fp16": FP16, "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "dataloader_pin_memory": DATALOADER_PIN_MEMORY,
    "enable_audio_download": ENABLE_AUDIO_DOWNLOAD,
    "actual_train_records": actual_train_records, "actual_validation_records": actual_validation_records,
    "actual_train_hours": actual_train_hours, "actual_validation_hours": actual_validation_hours,
}
(PATHS["artifacts"] / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

env_info = get_environment_info()
# Audit-only package versions (not part of data contract hash).
try:
    import accelerate, datasets, pyarrow, safetensors, tokenizers, numpy
    env_info["package_versions"] = {
        "torch": env_info.get("torch_version"),
        "transformers": env_info.get("transformers_version"),
        "accelerate": getattr(accelerate, "__version__", None),
        "datasets": getattr(datasets, "__version__", None),
        "pyarrow": getattr(pyarrow, "__version__", None),
        "safetensors": getattr(safetensors, "__version__", None),
        "tokenizers": getattr(tokenizers, "__version__", None),
        "numpy": getattr(numpy, "__version__", None),
    }
except Exception as _env_exc:
    env_info["package_versions_error"] = f"{type(_env_exc).__name__}: {_env_exc}"
env_info.update({
    "run_id": RUN_ID, "device": str(device),
    "mps_fallback_state": os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK"),
    "batch_size": PER_DEVICE_TRAIN_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_steps": MAX_STEPS,
    "actual_train_records": actual_train_records, "actual_validation_records": actual_validation_records,
    "actual_train_hours": actual_train_hours, "actual_validation_hours": actual_validation_hours,
    "train_samples_per_second": train_samples_per_second,
    "training_duration_seconds": training_duration,
    "train_runtime_seconds": train_runtime_seconds,
    "perf_counter_seconds": perf_counter_seconds,
    "duration_source": duration_source,
    "step_time_seconds": step_time,
    "total_pilot_duration_seconds": training_duration, "peak_memory_bytes": peak_memory_bytes,
})
(PATHS["artifacts"] / "environment.json").write_text(
    json.dumps(env_info, ensure_ascii=False, indent=2), encoding="utf-8")

# training_history.csv is ALWAYS written, even if training failed part-way.
if trainer is not None and getattr(trainer, "state", None) and getattr(trainer.state, "log_history", None):
    pd.DataFrame(trainer.state.log_history).to_csv(PATHS["artifacts"] / "training_history.csv", index=False)
else:
    pd.DataFrame(columns=["step", "loss", "eval_cer", "eval_wer"]).to_csv(
        PATHS["artifacts"] / "training_history.csv", index=False)

pd.DataFrame([
    {"check": "prerequisite_check", "passed": bool(prereq_check.get("passed"))},
    {"check": "tokenizer_provenance", "passed": bool(tokenizer_prov.get("passed"))},
    {"check": "data_contract_passed", "passed": bool(data_contract_passed)},
]).to_csv(PATHS["artifacts"] / "data_contract_checks.csv", index=False)

processor_prov = {
    "run_id": RUN_ID, "tokenizer_dir": str(PATHS["tokenizer"]),
    "vocab_size": (len(vocab) if vocab else None),
    "pad_token_id": (tokenizer.pad_token_id if tokenizer is not None else None),
    "unk_token_id": (tokenizer.unk_token_id if tokenizer is not None else None),
    "word_delimiter_id": vocab.get("|") if vocab else None,
    "feature_extractor_sampling_rate": (feature_extractor.sampling_rate if feature_extractor is not None else None),
    "model_id": PRETRAINED_MODEL_ID, "model_revision": PRETRAINED_MODEL_REVISION,
    "vocab_sha256": (sha256_file(PATHS["tokenizer"] / "vocab.json")
                     if (PATHS["tokenizer"] / "vocab.json").exists() else None),
}
(PATHS["artifacts"] / "processor_provenance.json").write_text(
    json.dumps(processor_prov, ensure_ascii=False, indent=2), encoding="utf-8")

oov_summary_df.to_csv(PATHS["artifacts"] / "validation_oov_summary.csv", index=False)
(PATHS["artifacts"] / "validation_oov_summary.json").write_text(json.dumps({
    "run_id": RUN_ID, "full_clean_validation_oov_chars": int(len(oov_full_df)),
    "active_validation_oov_chars": int(len(oov_active_df)),
    "records": oov_summary_df.to_dict(orient="records"),
}, ensure_ascii=False, indent=2), encoding="utf-8")

if cache_pool_summary:
    (PATHS["artifacts"] / "cache_pool_summary.json").write_text(
        json.dumps(cache_pool_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("config/environment/history/contract/provenance/oov/cache_pool written")


In [ ]:
# Cell 25 — (3/9) best_checkpoint_metadata.json (actual best step, not final global step)
best_ckpt_meta = {
    "run_id": RUN_ID, "model_id": PRETRAINED_MODEL_ID, "model_revision": PRETRAINED_MODEL_REVISION,
    "checkpoint_path": str(best_checkpoint_path) if best_checkpoint_path else None,
    "checkpoint_name": Path(best_checkpoint_path).name if best_checkpoint_path else None,
    "best_checkpoint_step": best_checkpoint_step,
    "final_global_step": getattr(train_result, "global_step", None) if training_success else None,
    "best_cer": best_cer, "best_metric": best_metric,
    "checkpoint_exists": bool(best_checkpoint_exists),
    "reload_ok": bool(checkpoint_reload_ok), "checkpoint_stage_ok": bool(checkpoint_stage_ok),
    "checkpoint_hash_note": "large weights not hashed to keep review lightweight",
}
(PATHS["artifacts"] / "best_checkpoint_metadata.json").write_text(
    json.dumps(best_ckpt_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("best_checkpoint_metadata.json written")


In [ ]:
# Cell 26 — (4/9) validation metrics + predictions (predictions.csv ALWAYS has header)
predictions_df = predictions_df.reindex(columns=VALIDATION_PREDICTIONS_COLUMNS)
predictions_df.to_csv(PATHS["artifacts"] / "validation_predictions.csv", index=False)
validation_metrics = {
    "run_id": RUN_ID, "run_mode": RUN_MODE, "training_success": training_success,
    "eval_metrics": eval_metrics, "n_predictions": int(len(predictions_df)),
    "mean_cer": float(predictions_df["cer"].astype(float).mean()) if len(predictions_df) else None,
    "mean_wer": float(predictions_df["wer"].astype(float).mean()) if len(predictions_df) else None,
    "validation_metrics_valid": bool(validation_metrics_valid),
}
(PATHS["artifacts"] / "validation_metrics.json").write_text(
    json.dumps(validation_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"validation_predictions.csv ({len(predictions_df)}) + validation_metrics.json written")


In [ ]:
# Cell 27 — (5/9) Frozen-test safety + final_integrity_checks.csv (+ cross-file run_id)
frozen_hits = detect_frozen_leakage(
    opened_paths=OPENED_PATHS, loaded_splits=LOADED_SPLITS,
    source_splits=(list(active_train_df.get("source_split", pd.Series([], dtype=str)).astype(str)) +
                   list(active_validation_df.get("source_split", pd.Series([], dtype=str)).astype(str))),
    split_values=(list(active_train_df.get("split", pd.Series([], dtype=str)).astype(str)) +
                  list(active_validation_df.get("split", pd.Series([], dtype=str)).astype(str))),
)
frozen_test_accessed = bool(frozen_hits)

# Cross-file RUN_ID consistency over everything written so far (run_summary/manifest excluded).
cross_file_issues = verify_artifacts_run_id([PATHS["artifacts"], PATHS["results"]], RUN_ID)
cross_file_run_id_consistent = (len(cross_file_issues) == 0)

integrity = [
    ("data_contract_passed", bool(data_contract_passed)),
    ("tokenizer_provenance_passed", bool(tokenizer_provenance_passed)),
    ("cache_pool_sufficient", bool(cache_pool_sufficient)),
    ("active_counts_correct", bool(active_counts_correct)),
    ("split_overlaps_zero", bool(overlaps_zero)),
    ("source_split_train_only", bool(source_split_ok)),
    ("ctc_feasibility_passed", bool(ctc_feasibility_passed)),
    ("gradient_smoke_test_passed", bool(smoke_test_passed)),
    ("training_completed", bool(training_success)),
    ("best_checkpoint_exists", bool(best_checkpoint_exists)),
    ("checkpoint_in_run_dir", bool(checkpoint_belongs_to_run(best_checkpoint_path, PATHS["checkpoints"]))),
    ("checkpoint_reload_ok", bool(checkpoint_reload_ok)),
    ("predictions_count_match", bool(predictions_count_match)),
    ("validation_metrics_valid", bool(validation_metrics_valid)),
    ("frozen_test_not_accessed", not frozen_test_accessed),
    ("cross_file_run_id_consistent", bool(cross_file_run_id_consistent)),
]
integrity_df = pd.DataFrame([{"check": n, "passed": bool(v)} for n, v in integrity])
integrity_df.to_csv(PATHS["artifacts"] / "final_integrity_checks.csv", index=False)
all_integrity_passed = bool(integrity_df["passed"].all())
print(integrity_df.to_string(index=False))
print(f"frozen_hits={frozen_hits} cross_file_issues={cross_file_issues} all_integrity_passed={all_integrity_passed}")


In [ ]:
# Cell 28 — (6/9) Status flags + (7/9) run_summary.json
pilot_checks = {
    "run_mode_pilot": RUN_MODE == "pilot",
    "cache_pool_sufficient": cache_pool_sufficient,
    "active_counts_correct": active_counts_correct,
    "data_contract_passed": data_contract_passed,
    "tokenizer_provenance_passed": tokenizer_provenance_passed,
    "frozen_test_not_accessed": frozen_test_accessed is False,
    "ctc_feasibility_passed": ctc_feasibility_passed,
    "smoke_test_passed": smoke_test_passed,
    "training_success": training_success,
    "best_checkpoint_exists": best_checkpoint_exists,
    "checkpoint_reload_ok": checkpoint_reload_ok,
    "predictions_count_match": predictions_count_match,
    "validation_metrics_valid": validation_metrics_valid,
    "all_integrity_passed": all_integrity_passed,
}
pilot_passed = all(pilot_checks.values())
if RUN_MODE == "pilot":
    STATUS = derive_pilot_status(RUN_MODE, pilot_passed)
    full_training_completed = False
    ready_for_rq1_final = False
    ready_for_notebook_04 = bool(pilot_passed)
else:
    # Full-mode: current-run errors win over durable summaries (fail-closed)
    STATUS = resolve_full_stage_status(
        pipeline_error=pipeline_error,
        current_full_status=full_status,
        full_stage=FULL_STAGE,
        state_dir=FULL_STATE_DIR,
        experiment_id=FULL_EXPERIMENT_ID,
        expected_contract=full_stage_contract,
    )
    if pipeline_error:
        STATUS = "FAILED"
    pilot_passed = False
    full_training_completed = (STATUS in {"SUCCESS_FULL_TRAINING", "SUCCESS_FULL_EVALUATE"})
    # NB04 handoff only after successful full train/evaluate of this run (not stale artifacts)
    ready_for_rq1_final = False  # RQ1 final needs frozen-test evaluation in later notebook
    ready_for_notebook_04 = bool(
        full_training_completed and pipeline_error is None and can_continue
        and (STATUS in {"SUCCESS_FULL_TRAINING", "SUCCESS_FULL_EVALUATE"})
    )

failure_reason = None
if STATUS == "FAILED":
    failed_checks = [k for k, v in pilot_checks.items() if not v]
    failure_reason = (f"stage={pipeline_error_stage}: {pipeline_error['message']}"
                      if pipeline_error else f"failed_checks={failed_checks}")

run_summary = {
    "run_id": RUN_ID, "run_mode": RUN_MODE, "status": STATUS,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_id": DATASET_ID, "dataset_revision": EXPECTED_DATASET_REVISION,
    "pretrained_model": PRETRAINED_MODEL_ID, "pretrained_model_revision": PRETRAINED_MODEL_REVISION,
    "device": str(device),
    "actual_train_records": actual_train_records, "actual_validation_records": actual_validation_records,
    "actual_train_hours": actual_train_hours, "actual_validation_hours": actual_validation_hours,
    "training_success": training_success,
    "training_duration_seconds": training_duration,
    "train_runtime_seconds": train_runtime_seconds,
    "perf_counter_seconds": perf_counter_seconds,
    "duration_source": duration_source,
    "step_time_seconds": step_time,
    "pipeline_error_stage": pipeline_error_stage, "failure_reason": failure_reason,
    "validation_metrics": eval_metrics,
    "best_checkpoint": str(best_checkpoint_path) if best_checkpoint_path else None,
    "best_checkpoint_step": best_checkpoint_step, "best_cer": best_cer,
    "pilot_checks": pilot_checks, "pilot_passed": pilot_passed,
    "full_training_implemented": FULL_TRAINING_IMPLEMENTED,
    "full_stage": FULL_STAGE if RUN_MODE == "full" else None,
    "full_experiment_id": FULL_EXPERIMENT_ID if RUN_MODE == "full" else None,
    "full_checkpoint_dir": str(experiment_checkpoint_dir(LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER)) if RUN_MODE == "full" else None,
    "full_durable_checkpoint_dir": str(experiment_checkpoint_dir(FULL_STATE_DIR / "checkpoints", FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER)) if RUN_MODE == "full" else None,
    "run_id": RUN_ID,
    "paths_artifacts": str(PATHS["artifacts"]),
    "paths_checkpoints": str(PATHS["checkpoints"]),
    "full_training_completed": full_training_completed,
    "ready_for_notebook_04": ready_for_notebook_04, "ready_for_rq1_final": ready_for_rq1_final,
    "frozen_test_accessed": frozen_test_accessed,
    "integrity_checks": {n: bool(v) for n, v in integrity},
}
(PATHS["results"] / "run_summary.json").write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
(PATHS["results_root"]).mkdir(parents=True, exist_ok=True)
(PATHS["results_root"] / "latest.json").write_text(json.dumps({
    "run_id": RUN_ID, "status": STATUS, "results_dir": str(PATHS["results"]),
    "artifacts_dir": str(PATHS["artifacts"]),
}, ensure_ascii=False, indent=2), encoding="utf-8")

# Invariant: SUCCESS_PILOT implies pilot_passed.
assert not (STATUS == "SUCCESS_PILOT" and not pilot_passed), "STATUS/pilot_passed invariant violated"
print(f"run_summary.json written (status={STATUS}, pilot_passed={pilot_passed})")


In [ ]:
# Cell 29 — (7/9) training_error.json on failure
if STATUS == "FAILED":
    err_payload = {
        "run_id": RUN_ID, "status": STATUS, "stage": pipeline_error_stage,
        "exception_type": (pipeline_error.get("exception_type") if pipeline_error else None),
        "message": (pipeline_error.get("message") if pipeline_error else failure_reason),
        "traceback": (pipeline_error.get("traceback") if pipeline_error else None),
        "device": str(device), "run_mode": RUN_MODE, "config": run_config,
        "training_duration_seconds": training_duration,
        "train_runtime_seconds": train_runtime_seconds,
        "perf_counter_seconds": perf_counter_seconds,
        "duration_source": duration_source,
        "step_time_seconds": step_time,
    }
    (PATHS["artifacts"] / "training_error.json").write_text(
        json.dumps(err_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"training_error.json written (stage={pipeline_error_stage})")
else:
    print("no failure -> training_error.json not written")


In [ ]:
# Cell 30 — (8/9) artifact_manifest.json (created LAST; does NOT hash itself)
manifest_path = (PATHS["artifacts"] / "artifact_manifest.json").resolve()
items = []
for base in [PATHS["artifacts"], PATHS["results"]]:
    for p in sorted(base.rglob("*")):
        if p.is_file() and p.resolve() != manifest_path:
            items.append({
                "path": str(p.relative_to(_PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })
manifest = {
    "run_id": RUN_ID, "run_mode": RUN_MODE, "status": STATUS,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "artifacts_dir": str(PATHS["artifacts"].relative_to(_PROJECT_ROOT)),
    "results_dir": str(PATHS["results"].relative_to(_PROJECT_ROOT)),
    "cross_file_run_id_consistent": bool(cross_file_run_id_consistent),
    "n_artifacts": len(items), "artifacts": items,
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"artifact_manifest.json: {len(items)} files across artifacts/ + results/ (self-excluded)")


In [ ]:
# Cell 31 — (9/9) Final status print (raises ONLY here if FAILED)
print("=" * 60)
print(f"NOTEBOOK 03 STATUS: {STATUS}")
print(f"RUN MODE: {RUN_MODE}   PILOT PASSED: {pilot_passed}")
print(f"FULL TRAINING IMPLEMENTED: {FULL_TRAINING_IMPLEMENTED}   COMPLETED: {full_training_completed}")
print(f"READY FOR NOTEBOOK 04: {ready_for_notebook_04}   READY FOR RQ1 FINAL: {ready_for_rq1_final}")
print(f"FROZEN TEST ACCESSED: {frozen_test_accessed}")
if STATUS != "SUCCESS_PILOT":
    print("Failed checks:", [k for k, v in pilot_checks.items() if not v])
print("=" * 60)

if STATUS == "FAILED":
    raise RuntimeError(
        f"NOTEBOOK 03 STATUS: FAILED (stage={pipeline_error_stage}, reason={failure_reason}). "
        f"See training_error.json in {PATHS['artifacts']}"
    )


In [ ]:
# Cell 32 — Export run artifacts to DURABLE_ROOT (uses RUN_ID / PATHS, no hardcoded run)
import shutil

Path(DURABLE_ROOT).mkdir(parents=True, exist_ok=True)
destination = Path(RUNTIME_PATHS.export_root) / RUN_ID
destination.mkdir(parents=True, exist_ok=True)

sources = {
    "artifacts": PATHS["artifacts"],
    "results": PATHS["results"],
}
# Full mode keeps checkpoints under the experiment root, not the per-run dir, and
# what exists at all depends on the stage: prepare and resume-test never produce a
# full-train checkpoint, so demanding one would fail an otherwise healthy run.
export_pointer = None
if RUN_MODE == "full":
    # Every full stage writes its durable product under FULL_STATE_DIR, which is
    # already on DURABLE_ROOT. Copying those bytes again would duplicate GBs outside
    # DURABLE_CHECKPOINT_BUDGET_BYTES, so the export records where they live.
    export_pointer = {
        "run_id": RUN_ID,
        "full_stage": FULL_STAGE,
        "experiment_id": FULL_EXPERIMENT_ID,
        "full_state_dir": str(FULL_STATE_DIR),
    }
    _required_summary = {
        "prepare": "full_data_summary.json",
        "train": "full_train_summary.json",
        "evaluate": "full_evaluate_summary.json",
    }.get(FULL_STAGE)
    if _required_summary:
        _summary_path = FULL_STATE_DIR / _required_summary
        if not _summary_path.is_file():
            raise RuntimeError(
                f"Refusing to export FULL_STAGE={FULL_STAGE!r} without {_summary_path}"
            )
        export_pointer["stage_summary"] = _required_summary
        export_pointer["stage_status"] = json.loads(
            _summary_path.read_text(encoding="utf-8")
        ).get("status")

    if FULL_STAGE in {"train", "evaluate"}:
        # Exporting an unvalidated checkpoint would publish an artifact NB04 cannot
        # trust, so the same gate as evaluate applies here.
        if not full_stage_contract:
            raise RuntimeError(
                f"Refusing to export FULL_STAGE={FULL_STAGE!r} without the canonical "
                "contract of this run"
            )
        _train_summary_path = FULL_STATE_DIR / "full_train_summary.json"
        if not _train_summary_path.is_file():
            raise RuntimeError(f"Refusing to export without {_train_summary_path}")
        _export_exp_dir = experiment_checkpoint_dir(
            LOCAL_CKPT_ROOT, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER
        )
        _export_best = resolve_best_checkpoint_from_durable(
            FULL_STATE_DIR,
            experiment_id=FULL_EXPERIMENT_ID,
            train_summary=json.loads(_train_summary_path.read_text(encoding="utf-8")),
            expected_contract=full_stage_contract,
            local_experiment_dir=_export_exp_dir,
        )
        print(f"export gate ok: best checkpoint {_export_best}")
        export_pointer["best_checkpoint"] = str(_export_best)
        export_pointer["local_experiment_dir"] = str(_export_exp_dir)
        export_pointer["durable_experiment_dir"] = str(durable_experiment_dir(
            FULL_STATE_DIR, FULL_EXPERIMENT_ID, kind=FULL_TRAIN_MARKER
        ))
    else:
        print(f"FULL_STAGE={FULL_STAGE!r}: no full-train checkpoint expected at this stage")
else:
    sources["checkpoints"] = PATHS["checkpoints"]

for name, source in sources.items():
    if not source.is_dir():
        raise RuntimeError(f"Missing export source {name}: {source}")
    shutil.copytree(source, destination / name, dirs_exist_ok=True)
    print(f"Copied: {name} <- {source}")

latest_json = PATHS["results_root"] / "latest.json"
if latest_json.is_file():
    shutil.copy2(latest_json, destination / "latest.json")

for required in ("artifacts/artifact_manifest.json", "results/run_summary.json"):
    if not (destination / required).is_file():
        raise RuntimeError(f"Export incomplete, missing {required} under {destination}")

if export_pointer is not None:
    (destination / "full_export_pointer.json").write_text(
        json.dumps(export_pointer, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Wrote full_export_pointer.json:", json.dumps(export_pointer, indent=2))

if "checkpoints" in sources:
    checkpoints = sorted((destination / "checkpoints").glob("checkpoint-*"))
    if not checkpoints:
        raise RuntimeError(f"No checkpoint-* copied into {destination / 'checkpoints'}")
    print("Checkpoints:", [p.name for p in checkpoints])

print("Saved successfully:", destination)


In [ ]:
# Removed: legacy ad-hoc audit cell (removed).
# Use FULL_STATE_DIR from RUNTIME_PATHS.prepare_state_dir(contract_hash).


In [ ]:
# Removed: legacy ad-hoc audit cell (removed).
# Use FULL_STATE_DIR from RUNTIME_PATHS.prepare_state_dir(contract_hash).


In [ ]:
# Cleared: ad-hoc duration histogram (depends on prepare-only locals).


## Hand-off sang Notebook 04

- `STATUS = SUCCESS_PILOT` (⇔ `pilot_passed = True`) nghĩa là **pipeline ASR pilot** chạy được
  với cache pool Notebook 02 — chưa phải kết quả full-training luận văn.
- Full-mode: `STATUS` lấy từ `resolve_full_stage_status` (ưu tiên `pipeline_error` phiên hiện tại;
  artifact SUCCESS cũ không được che lỗi mới).
- `ready_for_notebook_04 = True` chỉ khi full-mode đạt `SUCCESS_FULL_TRAINING` hoặc
  `SUCCESS_FULL_EVALUATE` trong phiên hiện tại (không lỗi pipeline).
- `ready_for_rq1_final` vẫn `False` ở Notebook 03 — RQ1 final cần frozen-test evaluation riêng.
- Artifacts: `artifacts/notebook03/<RUN_ID>/` và `results/notebook03/<RUN_ID>/`.
- Frozen test tuyệt đối không được truy cập ở Notebook 03.
